# Set up

In [1]:
# load libraries
import pandas as pd
from census import Census
from us import states
import requests
import time
import numpy as np 

# load functions
from analysis_functions import (
    fetch_census_data,
    fetch_census_data_msa_acs5,
    fetch_acs5_counties_tx,
    create_pivot_tables,
    print_rankings,
)


From Jess: 
I'd like an idea of 3 things and I'm not sure how they should be represented (I'll leave that to you)
- Population growth over the last 10, 15, and 20 years
- Migration to and from the city over that timeframe
- Median age of the population over that time frame

In [2]:
# set census api key 
c = Census("8149a6071641f66c9a2eaa2a679734a3d2a148e1")
# want data from the past 5, 10, 15 and 20 years
years = list(range(2004, 2025))

years_acs5 = [2009, 2014, 2019, 2024]  # Non-overlapping 5-year periods

# define the variables we need from the census api
census_variables = (
    'NAME', 
    'B01003_001E',  # total population
    'B11001_002E',  # total families
    # 'B11003_004E',  # families with only children under 6
    # 'B11003_005E',  # families with children under 6 AND children 6-17
    'B11005_003E',  # family households with one or more people under 18
    'B01002_001E',  # median age
    'B09001_001E',  # population under 18
    'B07003_004E',  # lived in the same house 1 year ago
    'B07003_007E',  # moved within same county
    'B07003_010E',  # moved from a different county in the same state
    'B07003_013E',  # moved from a different state
    'B07003_016E',  # moved from abroad
)

## Get ACS1 data

In [3]:
# get msa data 
df_msa = fetch_census_data(
    census_obj=c,
    geography='msa',
    years=years,
    variables=census_variables,
    include_moe=True,
)

Error for year 2004: <!doctype html><html lang="en"><head><title>HTTP Status 404 ? Not Found</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 404 ? Not Found</h1></body></html>
Retrieved msa data for 2005: 500 records
Retrieved msa data for 2006: 507 records
Retrieved msa data for 2007: 510 records
Retrieved msa data for 2008: 512 records
Retrieved msa data for 2009: 517 records
Retrieved msa data for 2010: 525 records
Retrieved msa data for 2011: 529 records
Retrieved msa data for 2012: 530 records
Retrieved msa data for 2013: 515 records
Retrieved msa data for 2014: 515 records
Retrieved msa data for 2015: 516 records
Retrieved msa data for 2016: 518 records
Retrieved msa data for 2017: 518 records
Retrieved msa data

In [4]:
# get county data 
tx_counties = fetch_census_data(
    census_obj=c,
    geography='county',
    years=years,
    variables=census_variables,
    include_moe=True,
)

Error for year 2004: Geography is not available in 2004. Available years include (2024, 2023, 2022, 2021, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005)
Retrieved county data for 2005: 49 records
Retrieved county data for 2006: 50 records
Retrieved county data for 2007: 50 records
Retrieved county data for 2008: 50 records
Retrieved county data for 2009: 50 records
Retrieved county data for 2010: 52 records
Retrieved county data for 2011: 53 records
Retrieved county data for 2012: 53 records
Retrieved county data for 2013: 53 records
Retrieved county data for 2014: 53 records
Retrieved county data for 2015: 53 records
Retrieved county data for 2016: 53 records
Retrieved county data for 2017: 54 records
Retrieved county data for 2018: 54 records
Retrieved county data for 2019: 54 records
Error for year 2020: Geography is not available in 2020. Available years include (2024, 2023, 2022, 2021, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011,

In [5]:
df_msa.head(10)

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,family_households_with_child_moe,median_age_moe,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,msa_code,year
0,"Decatur, AL Metro Area",146379.0,40614.0,19783.0,38.7,34840.0,121425.0,15794.0,4548.0,2837.0,...,1719.0,0.8,267.0,4185.0,3430.0,2042.0,1423.0,143.0,19460,2005
1,"Decatur, IL Metro Area",106433.0,29734.0,14431.0,39.3,26353.0,87475.0,12612.0,2093.0,2159.0,...,1399.0,0.7,0.0,2799.0,2498.0,897.0,1199.0,247.0,19500,2005
2,"Deltona-Daytona Beach-Ormond Beach, FL Metro Area",475189.0,127674.0,55714.0,42.7,98163.0,389550.0,45685.0,12061.0,21096.0,...,3331.0,0.3,278.0,8572.0,6136.0,3346.0,4083.0,807.0,19660,2005
3,"Denver-Aurora, CO Metro Area",2327901.0,585045.0,310967.0,34.6,614480.0,1837272.0,214141.0,152498.0,66123.0,...,6891.0,0.2,1241.0,16847.0,12462.0,11601.0,7801.0,3212.0,19740,2005
4,"Des Moines, IA Metro Area",511565.0,140049.0,74182.0,36.0,126951.0,414264.0,55379.0,15509.0,17388.0,...,3208.0,0.3,1250.0,7625.0,6868.0,2699.0,4132.0,549.0,19780,2005
5,"Detroit-Warren-Livonia, MI Metro Area",4428941.0,1136763.0,593643.0,36.9,1156653.0,3779007.0,423897.0,104597.0,41373.0,...,9569.0,0.2,340.0,23014.0,17958.0,9599.0,4642.0,3215.0,19820,2005
6,"Dothan, AL Metro Area",134993.0,38194.0,18616.0,39.3,32693.0,113087.0,12421.0,3043.0,4447.0,...,1376.0,0.8,216.0,3110.0,2483.0,1043.0,1209.0,327.0,20020,2005
7,"Dover, DE Metro Area",140205.0,38719.0,21289.0,35.7,35980.0,117523.0,14838.0,2341.0,3766.0,...,1412.0,0.4,329.0,3436.0,3048.0,1176.0,1129.0,473.0,20100,2005
8,"DuBois, PA Micro Area",79001.0,21874.0,9636.0,41.4,16883.0,69391.0,4192.0,3327.0,1193.0,...,898.0,0.5,146.0,1890.0,1299.0,1555.0,574.0,84.0,20180,2005
9,"Dubuque, IA Metro Area",86626.0,24048.0,11760.0,39.2,NaN,75426.0,6895.0,984.0,1800.0,...,1212.0,0.6,NaN,2139.0,1963.0,396.0,939.0,119.0,20220,2005


In [6]:
tx_counties.head(10)

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,median_age_moe,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,state,county_code,year
0,"Angelina County, Texas",78839.0,21098.0,11364.0,34.7,22099.0,61109.0,11790.0,3747.0,1329.0,...,0.6,0.0,3730.0,3268.0,2253.0,811.0,151.0,48,005,2005
1,"Bastrop County, Texas",61724.0,14862.0,6777.0,37.9,NaN,54507.0,2852.0,1933.0,623.0,...,3.9,NaN,4896.0,1444.0,945.0,624.0,519.0,48,021,2005
2,"Bell County, Texas",246592.0,66060.0,38428.0,30.2,80298.0,174071.0,30530.0,7468.0,15354.0,...,0.3,569.0,6457.0,5680.0,2384.0,3484.0,4452.0,48,027,2005
3,"Bexar County, Texas",1482598.0,357432.0,204070.0,33.0,422116.0,1172655.0,195524.0,36315.0,38089.0,...,0.2,239.0,16432.0,14916.0,5590.0,7042.0,5176.0,48,029,2005
4,"Bowie County, Texas",83813.0,24008.0,12827.0,36.9,21682.0,67032.0,9319.0,2425.0,4339.0,...,0.8,34.0,2892.0,2590.0,1051.0,1534.0,190.0,48,037,2005
5,"Brazoria County, Texas",267376.0,71863.0,41549.0,33.4,76373.0,210248.0,32282.0,15294.0,4140.0,...,0.3,100.0,8033.0,7166.0,3538.0,1962.0,1047.0,48,039,2005
6,"Brazos County, Texas",143502.0,30397.0,16378.0,25.0,32505.0,105835.0,18150.0,11982.0,2812.0,...,0.3,0.0,4153.0,2902.0,2794.0,1363.0,1165.0,48,041,2005
7,"Cameron County, Texas",374081.0,89799.0,55828.0,28.5,128146.0,312733.0,39556.0,5340.0,6930.0,...,0.3,231.0,8703.0,8441.0,2024.0,2784.0,770.0,48,061,2005
8,"Collin County, Texas",655994.0,165649.0,96710.0,33.6,181559.0,525074.0,48553.0,43116.0,19709.0,...,0.3,0.0,9200.0,7360.0,6801.0,4559.0,2578.0,48,085,2005
9,"Comal County, Texas",94794.0,25606.0,11687.0,36.7,NaN,79775.0,5169.0,7275.0,874.0,...,1.1,NaN,3482.0,2316.0,2526.0,624.0,786.0,48,091,2005


## Get ACS5 data
Data only available from 2009-2023 at the county level, and 2009-2024 at the MSA level.

In [7]:
# get msa data using ACS 5-year estimates
df_msa_acs5 = fetch_census_data_msa_acs5(
    census_obj=c,
    geography='msa',
    years=years_acs5,  # Use non-overlapping years only
    variables=census_variables,
    include_moe=True,
)

Retrieved msa data for 2009 (ACS5): 953 records
Retrieved msa data for 2014 (ACS5): 929 records
Retrieved msa data for 2019 (ACS5): 938 records
Retrieved msa data for 2024 (ACS5): 935 records


In [8]:
# get county data using ACS 5-year 
# for the counties data, the census package hasn't updated for the new five-year estimates yet.
# instead, we'll pull straight from the api using the requests package

frames = []

for y in years_acs5:
    df_year = fetch_acs5_counties_tx(y, include_moe=True)
    frames.append(df_year)
    
    time.sleep(1)  # be polite to Census API

tx_counties_acs5 = pd.concat(frames, ignore_index=True)

cols = [
    'B01003_001E',
    'B11001_002E',
    'B11005_003E',
    'B01002_001E',
    'B09001_001E',
    'B07003_004E',
    'B07003_007E',
    'B07003_010E',
    'B07003_013E',
    'B07003_016E',
    'B01003_001M',
    'B11001_002M',
    'B11005_003M',
    'B01002_001M',
    'B09001_001M',
    'B07003_004M',
    'B07003_007M',
    'B07003_010M',
    'B07003_013M',
    'B07003_016M'
]

tx_counties_acs5[cols] = tx_counties_acs5[cols].apply(
    pd.to_numeric, errors="coerce"
)

In [9]:
# rename colums for clarity
county_mapping = {
        'NAME': 'name',
        'B01003_001E': 'population',
        'B11001_002E': 'total_families',
        'B11005_003E': 'family_households_with_child',
        'B01002_001E': 'median_age',
        'B09001_001E': 'population_under_18',
        'B07003_004E': 'lived_in_same_house',
        'B07003_007E': 'moved_within_county',
        'B07003_010E': 'moved_from_diff_county_same_state',
        'B07003_013E': 'moved_from_diff_state',
        'B07003_016E': 'moved_from_abroad',
        'B01003_001M': 'population_moe',
        'B11001_002M': 'total_families_moe',
        'B11005_003M': 'family_households_with_child_moe',
        'B01002_001M': 'median_age_moe',
        'B09001_001M': 'population_under_18_moe',
        'B07003_004M': 'lived_in_same_house_moe',
        'B07003_007M': 'moved_within_county_moe',
        'B07003_010M': 'moved_from_diff_county_same_state_moe',
        'B07003_013M': 'moved_from_diff_state_moe',
        'B07003_016M': 'moved_from_abroad_moe',
        'county': 'county_code'
    }

tx_counties_acs5 = tx_counties_acs5.rename(columns=county_mapping)

In [10]:
# check out the data
tx_counties_acs5.head(10)

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,median_age_moe,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,state,county_code,year
0,"Burnet County, Texas",43402,11242,4708,44.7,9835,37013,2884,2496,610,...,0.2,139,845,697,616,282,30,48,053,2009
1,"Caldwell County, Texas",36895,8280,4437,35.8,9917,29714,2333,3995,469,...,0.5,64,1243,499,1080,247,37,48,055,2009
2,"Calhoun County, Texas",20384,5479,2896,38.1,5476,16744,1989,946,345,...,0.7,38,585,519,258,237,136,48,057,2009
3,"Callahan County, Texas",13300,3686,1645,41.0,3185,11494,578,1000,90,...,0.8,100,455,262,354,60,127,48,059,2009
4,"Cameron County, Texas",383171,93322,58790,29.1,133416,329375,32918,4970,4355,...,0.1,97,3032,2732,757,693,616,48,061,2009
5,"Camp County, Texas",12489,3301,1635,36.5,3482,10887,631,570,129,...,1.5,0,500,348,252,110,72,48,063,2009
6,"Carson County, Texas",6246,1863,724,41.7,1589,5221,360,494,76,...,1.5,21,223,112,200,64,127,48,065,2009
7,"Cass County, Texas",29315,8772,4023,43.2,6715,25308,2235,941,466,...,0.5,5,577,482,246,149,26,48,067,2009
8,"Castro County, Texas",7233,2228,1259,35.7,2216,6095,576,322,46,...,0.6,0,264,205,151,51,46,48,069,2009
9,"Chambers County, Texas",29198,9158,5235,37.1,8113,24862,1463,2088,312,...,0.5,31,636,384,622,171,6,48,071,2009


# Texas MSA analysis 

## ACS 1-year

In [11]:
# filter for metro areas in texas only 
msa_tx = df_msa[df_msa['name'].str.contains('TX', case=False, na=False)].copy()

# some msa_codes have multiple names
# fix that by aggregating instances where there are multiple rows for the same metro area for the same year
# pick just the most common name
msa_tx['name'] = msa_tx.groupby('msa_code')['name'].transform(lambda x: x.mode()[0])

# create column for all people who moved to the area from a different location
msa_tx['total_in_migration'] = (
    msa_tx['moved_from_diff_county_same_state'] + 
    msa_tx['moved_from_diff_state'] + 
    msa_tx['moved_from_abroad']
)

# aggregate to ensure one row per msa_code per year
msa_tx_agg = msa_tx.groupby(['msa_code', 'name', 'year']).agg({
    'population': 'sum',
    'total_families': 'sum',
    'median_age': 'median',
    'population_under_18': 'sum',
    'family_households_with_child': 'sum'
}).reset_index()

print(msa_tx['msa_code'].nunique())

33


In [12]:
# use function to create multiple dfs at once, pivoting from original so each column is a year
msa_pivots = create_pivot_tables(
    df=msa_tx_agg,
    index_cols=['msa_code','name'],
    variable_names=['population', 
                    'median_age', 
                    'total_families', 
                    'population_under_18', 
                    'family_households_with_child'])

# extract individual pivot tables
msa_tx_pop = msa_pivots['population']
msa_tx_age = msa_pivots['median_age']
msa_tx_families = msa_pivots['total_families']
msa_tx_under18 = msa_pivots['population_under_18']
msa_tx_family_households_with_child = msa_pivots['family_households_with_child']

# create df specifically for migration
# this one is a little different because there are multiple values 
msa_tx_migration = msa_tx.pivot(
    index=['msa_code','name'],
    columns='year',
    values=['lived_in_same_house', 
            'moved_within_county', 
            'moved_from_diff_county_same_state',
            'moved_from_diff_state',
            'moved_from_abroad',
            'total_in_migration']
).reset_index()

# flatten cols so theyre not stacked
msa_tx_migration.columns = [
    f"{var}_{year}" for var, year in msa_tx_migration.columns
]

# reset index
msa_tx_migration = msa_tx_migration.reset_index();

### Population change

In [13]:
# create columns for population change in the past 5, 10, 15 and 19 years (2005–2024)
msa_tx_pop['pct_change_19_24'] = ((msa_tx_pop[2024] - msa_tx_pop[2019]) / msa_tx_pop[2019]) * 100 # 5 years
msa_tx_pop['pct_change_14_24'] = ((msa_tx_pop[2024] - msa_tx_pop[2014]) / msa_tx_pop[2014]) * 100 # 10 years
msa_tx_pop['pct_change_09_24'] = ((msa_tx_pop[2024] - msa_tx_pop[2009]) / msa_tx_pop[2009]) * 100 # 15 years
msa_tx_pop['pct_change_05_24'] = ((msa_tx_pop[2024] - msa_tx_pop[2005]) / msa_tx_pop[2005]) * 100 # 19 years

In [14]:
# print ranked results
print_rankings(
    df=msa_tx_pop,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_05_24
0                    Austin-Round Rock, TX Metro Area         81.363929
1                College Station-Bryan, TX Metro Area         58.756102
2                              Midland, TX Metro Area         57.092382
3                       Killeen-Temple, TX Metro Area         54.193287
4     Houston-The Woodlands-Sugar Land, TX Metro Area         50.115723
5            San Antonio-New Braunfels, TX Metro Area         49.836173
6                             Longview, TX Metro Area         48.176936
7                              Lubbock, TX Metro Area         46.643631
8          Dallas-Fort Worth-Arlington, TX Metro Area         45.686439
9                                 Waco, TX Metro Area         42.516148
10                              Odessa, TX Metro Area         37.858284
11            McAllen-Edinburg-Mission, TX Metro Area         36.140614
12                               Tyler, TX Metro Area         34

{'pct_change_05_24': year                                             name  pct_change_05_24
 0                    Austin-Round Rock, TX Metro Area         81.363929
 1                College Station-Bryan, TX Metro Area         58.756102
 2                              Midland, TX Metro Area         57.092382
 3                       Killeen-Temple, TX Metro Area         54.193287
 4     Houston-The Woodlands-Sugar Land, TX Metro Area         50.115723
 5            San Antonio-New Braunfels, TX Metro Area         49.836173
 6                             Longview, TX Metro Area         48.176936
 7                              Lubbock, TX Metro Area         46.643631
 8          Dallas-Fort Worth-Arlington, TX Metro Area         45.686439
 9                                 Waco, TX Metro Area         42.516148
 10                              Odessa, TX Metro Area         37.858284
 11            McAllen-Edinburg-Mission, TX Metro Area         36.140614
 12                            

### Total number of families

In [15]:
# create columns for percentage change in the number of families
msa_tx_families['pct_change_19_24'] = ((msa_tx_families[2024] - msa_tx_families[2019]) / msa_tx_families[2019]) * 100 # 5 years
msa_tx_families['pct_change_14_24'] = ((msa_tx_families[2024] - msa_tx_families[2014]) / msa_tx_families[2014]) * 100 # 10 years
msa_tx_families['pct_change_09_24'] = ((msa_tx_families[2024] - msa_tx_families[2009]) / msa_tx_families[2009]) * 100 # 15 years
msa_tx_families['pct_change_05_24'] = ((msa_tx_families[2024] - msa_tx_families[2005]) / msa_tx_families[2005]) * 100 # 19 years

In [16]:
# print ranked results
print_rankings(
    df=msa_tx_families,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_05_24
0                    Austin-Round Rock, TX Metro Area         90.551594
1                              Midland, TX Metro Area         64.682253
2                College Station-Bryan, TX Metro Area         56.355436
3                             Longview, TX Metro Area         52.803703
4            San Antonio-New Braunfels, TX Metro Area         48.742176
5     Houston-The Woodlands-Sugar Land, TX Metro Area         48.538529
6                                 Waco, TX Metro Area         47.164852
7                       Killeen-Temple, TX Metro Area         45.392262
8                               Odessa, TX Metro Area         45.382840
9          Dallas-Fort Worth-Arlington, TX Metro Area         43.841879
10                               Tyler, TX Metro Area         32.100432
11            McAllen-Edinburg-Mission, TX Metro Area         31.659117
12                     Sherman-Denison, TX Metro Area         31

{'pct_change_05_24': year                                             name  pct_change_05_24
 0                    Austin-Round Rock, TX Metro Area         90.551594
 1                              Midland, TX Metro Area         64.682253
 2                College Station-Bryan, TX Metro Area         56.355436
 3                             Longview, TX Metro Area         52.803703
 4            San Antonio-New Braunfels, TX Metro Area         48.742176
 5     Houston-The Woodlands-Sugar Land, TX Metro Area         48.538529
 6                                 Waco, TX Metro Area         47.164852
 7                       Killeen-Temple, TX Metro Area         45.392262
 8                               Odessa, TX Metro Area         45.382840
 9          Dallas-Fort Worth-Arlington, TX Metro Area         43.841879
 10                               Tyler, TX Metro Area         32.100432
 11            McAllen-Edinburg-Mission, TX Metro Area         31.659117
 12                     Sherman

### Median age

In [17]:
# for median age, calculate absolute change 
msa_tx_age['change_19_24'] = (msa_tx_age[2024] - msa_tx_age[2019])  # 5 years
msa_tx_age['change_14_24'] = (msa_tx_age[2024] - msa_tx_age[2014])  # 10 years
msa_tx_age['change_09_24'] = (msa_tx_age[2024] - msa_tx_age[2009])  # 15 years
msa_tx_age['change_05_24'] = (msa_tx_age[2024] - msa_tx_age[2005])  # 19 years

In [18]:
# rank by cities that got the youngest
print("Metros by median age change (2005-2024):")
print(msa_tx_age[msa_tx_age['change_05_24'].notna()]
      .sort_values(by='change_05_24', ascending=True)
      .reset_index(drop=True)[['name', 'change_05_24', 2005, 2024]])

Metros by median age change (2005-2024):
year                                             name  change_05_24  2005  \
0                              Midland, TX Metro Area          -1.6  34.7   
1                              Abilene, TX Metro Area          -0.2  34.9   
2                               Odessa, TX Metro Area          -0.1  31.8   
3                        Wichita Falls, TX Metro Area          -0.1  36.4   
4                             Longview, TX Metro Area           1.2  37.2   
5                           San Angelo, TX Metro Area           1.2  34.8   
6                 Beaumont-Port Arthur, TX Metro Area           1.2  36.9   
7                              Lubbock, TX Metro Area           1.7  30.9   
8                      Sherman-Denison, TX Metro Area           1.8  37.5   
9                College Station-Bryan, TX Metro Area           1.9  26.8   
10                               Tyler, TX Metro Area           2.1  35.4   
11                            Amari

### Total population under 18

In [19]:
# percentage change
msa_tx_under18['pct_change_19_24'] = ((msa_tx_under18[2024] - msa_tx_under18[2019]) / msa_tx_under18[2019]) * 100 # 5 years
msa_tx_under18['pct_change_14_24'] = ((msa_tx_under18[2024] - msa_tx_under18[2014]) / msa_tx_under18[2014]) * 100 # 10 years
msa_tx_under18['pct_change_09_24'] = ((msa_tx_under18[2024] - msa_tx_under18[2009]) / msa_tx_under18[2009]) * 100 # 15 years
msa_tx_under18['pct_change_05_24'] = ((msa_tx_under18[2024] - msa_tx_under18[2005]) / msa_tx_under18[2005]) * 100 # 19 years

In [20]:
# print ranked results
print_rankings(
    df=msa_tx_under18,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_05_24
0                           San Angelo, TX Metro Area               inf
1                              Midland, TX Metro Area         61.562085
2                    Austin-Round Rock, TX Metro Area         50.022222
3                College Station-Bryan, TX Metro Area         38.554790
4                               Odessa, TX Metro Area         38.516101
5                             Longview, TX Metro Area         36.361208
6     Houston-The Woodlands-Sugar Land, TX Metro Area         33.021494
7                              Lubbock, TX Metro Area         32.308208
8            San Antonio-New Braunfels, TX Metro Area         27.295813
9                      Sherman-Denison, TX Metro Area         26.759213
10         Dallas-Fort Worth-Arlington, TX Metro Area         26.216238
11                      Killeen-Temple, TX Metro Area         24.621258
12                                Waco, TX Metro Area         23

{'pct_change_05_24': year                                             name  pct_change_05_24
 0                           San Angelo, TX Metro Area               inf
 1                              Midland, TX Metro Area         61.562085
 2                    Austin-Round Rock, TX Metro Area         50.022222
 3                College Station-Bryan, TX Metro Area         38.554790
 4                               Odessa, TX Metro Area         38.516101
 5                             Longview, TX Metro Area         36.361208
 6     Houston-The Woodlands-Sugar Land, TX Metro Area         33.021494
 7                              Lubbock, TX Metro Area         32.308208
 8            San Antonio-New Braunfels, TX Metro Area         27.295813
 9                      Sherman-Denison, TX Metro Area         26.759213
 10         Dallas-Fort Worth-Arlington, TX Metro Area         26.216238
 11                      Killeen-Temple, TX Metro Area         24.621258
 12                            

### Number of family households with children under 18

In [21]:
# calculate percentage changes for family households with at least one kid under 18
msa_tx_family_households_with_child['pct_change_19_24'] = ((msa_tx_family_households_with_child[2024] - msa_tx_family_households_with_child[2019]) / msa_tx_family_households_with_child[2019]) * 100 # 5 years
msa_tx_family_households_with_child['pct_change_14_24'] = ((msa_tx_family_households_with_child[2024] - msa_tx_family_households_with_child[2014]) / msa_tx_family_households_with_child[2014]) * 100 # 10 years
msa_tx_family_households_with_child['pct_change_09_24'] = ((msa_tx_family_households_with_child[2024] - msa_tx_family_households_with_child[2009]) / msa_tx_family_households_with_child[2009]) * 100 # 15 years
msa_tx_family_households_with_child['pct_change_05_24'] = ((msa_tx_family_households_with_child[2024] - msa_tx_family_households_with_child[2005]) / msa_tx_family_households_with_child[2005]) * 100 # 19 years

In [22]:
# print ranked results
print_rankings(
    df=msa_tx_family_households_with_child,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_05_24
0                    Austin-Round Rock, TX Metro Area         64.811644
1                              Midland, TX Metro Area         56.950524
2                College Station-Bryan, TX Metro Area         41.657448
3                             Longview, TX Metro Area         39.013238
4                               Odessa, TX Metro Area         36.717834
5                       Killeen-Temple, TX Metro Area         33.666276
6            San Antonio-New Braunfels, TX Metro Area         30.291777
7     Houston-The Woodlands-Sugar Land, TX Metro Area         30.273362
8          Dallas-Fort Worth-Arlington, TX Metro Area         23.619685
9                      Sherman-Denison, TX Metro Area         16.031056
10                               Tyler, TX Metro Area         14.977330
11                                Waco, TX Metro Area         14.476040
12                              Athens, TX Micro Area          9

{'pct_change_05_24': year                                             name  pct_change_05_24
 0                    Austin-Round Rock, TX Metro Area         64.811644
 1                              Midland, TX Metro Area         56.950524
 2                College Station-Bryan, TX Metro Area         41.657448
 3                             Longview, TX Metro Area         39.013238
 4                               Odessa, TX Metro Area         36.717834
 5                       Killeen-Temple, TX Metro Area         33.666276
 6            San Antonio-New Braunfels, TX Metro Area         30.291777
 7     Houston-The Woodlands-Sugar Land, TX Metro Area         30.273362
 8          Dallas-Fort Worth-Arlington, TX Metro Area         23.619685
 9                      Sherman-Denison, TX Metro Area         16.031056
 10                               Tyler, TX Metro Area         14.977330
 11                                Waco, TX Metro Area         14.476040
 12                            

### Migration to Texas MSAs

In [23]:
# Calculate migration percentage changes
msa_tx_migration['pct_chg_migration_05_24'] = ((msa_tx_migration['total_in_migration_2024'] - msa_tx_migration['total_in_migration_2005']) / msa_tx_migration['total_in_migration_2005'] * 100)
msa_tx_migration['pct_chg_migration_09_24'] = ((msa_tx_migration['total_in_migration_2024'] - msa_tx_migration['total_in_migration_2009']) / msa_tx_migration['total_in_migration_2009'] * 100)
msa_tx_migration['pct_chg_migration_14_24'] = ((msa_tx_migration['total_in_migration_2024'] - msa_tx_migration['total_in_migration_2014']) / msa_tx_migration['total_in_migration_2014'] * 100)
msa_tx_migration['pct_chg_migration_19_24'] = ((msa_tx_migration['total_in_migration_2024'] - msa_tx_migration['total_in_migration_2019']) / msa_tx_migration['total_in_migration_2019'] * 100)

In [24]:
# Print ranked results
print_rankings(
    df=msa_tx_migration,
    name_col='name_',
    variable_names=['pct_chg_migration_05_24', 'pct_chg_migration_09_24',
                 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

                                              name_  pct_chg_migration_05_24
0                            Midland, TX Metro Area               248.428661
1                             Odessa, TX Metro Area               199.605263
2                            Lubbock, TX Metro Area                96.150691
3              College Station-Bryan, TX Metro Area                79.185659
4                  Austin-Round Rock, TX Metro Area                65.322141
5                           Amarillo, TX Metro Area                53.519363
6                            El Paso, TX Metro Area                51.352402
7          San Antonio-New Braunfels, TX Metro Area                48.938003
8        Dallas-Fort Worth-Arlington, TX Metro Area                43.363235
9                    Sherman-Denison, TX Metro Area                40.719446
10                              Waco, TX Metro Area                32.143328
11                        San Angelo, TX Metro Area                31.882936

{'pct_chg_migration_05_24':                                               name_  pct_chg_migration_05_24
 0                            Midland, TX Metro Area               248.428661
 1                             Odessa, TX Metro Area               199.605263
 2                            Lubbock, TX Metro Area                96.150691
 3              College Station-Bryan, TX Metro Area                79.185659
 4                  Austin-Round Rock, TX Metro Area                65.322141
 5                           Amarillo, TX Metro Area                53.519363
 6                            El Paso, TX Metro Area                51.352402
 7          San Antonio-New Braunfels, TX Metro Area                48.938003
 8        Dallas-Fort Worth-Arlington, TX Metro Area                43.363235
 9                    Sherman-Denison, TX Metro Area                40.719446
 10                              Waco, TX Metro Area                32.143328
 11                        San Angelo

## ACS1 margins of error

In [25]:
# Filter just for Tyler, TX
tyler_estimates_acs1 = msa_tx[msa_tx['name'] == 'Tyler, TX Metro Area'].copy()

tyler_estimates_acs1

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,median_age_moe,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,msa_code,year,total_in_migration
346,"Tyler, TX Metro Area",185465.0,48152.0,26026.0,35.4,49255.0,144741.0,20038.0,12408.0,3638.0,...,0.2,177.0,4081.0,3858.0,2456.0,1629.0,540.0,46340,2005,16814.0
969,"Tyler, TX Metro Area",194635.0,48471.0,24303.0,35.2,49980.0,152096.0,24363.0,7647.0,4786.0,...,0.2,279.0,5212.0,4351.0,1975.0,1536.0,1034.0,46340,2006,13726.0
1025,"Tyler, TX Metro Area",198705.0,48396.0,24532.0,35.3,50624.0,154608.0,25920.0,10017.0,4053.0,...,0.3,251.0,5101.0,4337.0,2316.0,1380.0,432.0,46340,2007,14573.0
1971,"Tyler, TX Metro Area",201277.0,48756.0,24358.0,35.8,50898.0,161619.0,23302.0,10093.0,3134.0,...,0.5,329.0,5316.0,4955.0,2644.0,1990.0,703.0,46340,2008,14059.0
2514,"Tyler, TX Metro Area",204665.0,47517.0,21899.0,36.1,52773.0,165461.0,20022.0,11575.0,3633.0,...,0.6,0.0,4666.0,4029.0,3036.0,1409.0,627.0,46340,2009,15956.0
2739,"Tyler, TX Metro Area",210512.0,53601.0,26382.0,35.5,54236.0,166384.0,26636.0,10171.0,2857.0,...,0.3,524.0,5540.0,5242.0,2945.0,1916.0,1016.0,46340,2010,14361.0
3414,"Tyler, TX Metro Area",213381.0,53407.0,25992.0,35.6,54404.0,165910.0,25906.0,12872.0,6230.0,...,0.3,292.0,6348.0,4968.0,3165.0,2931.0,252.0,46340,2011,19403.0
3971,"Tyler, TX Metro Area",214821.0,54091.0,26429.0,36.0,54602.0,170460.0,21024.0,14251.0,5551.0,...,0.4,293.0,6012.0,4227.0,4031.0,2173.0,553.0,46340,2012,20525.0
4606,"Tyler, TX Metro Area",216080.0,53610.0,27734.0,36.2,54209.0,174730.0,22026.0,9675.0,5403.0,...,0.5,<NA>,6449.0,4986.0,2275.0,2022.0,313.0,46340,2013,15526.0
4972,"Tyler, TX Metro Area",218842.0,53239.0,25000.0,36.3,54587.0,178721.0,24562.0,8009.0,3510.0,...,0.6,57.0,5087.0,4560.0,2168.0,1540.0,519.0,46340,2014,12223.0


In [26]:
# Create migration confidence intervals dataframe for Smith County
migration_data_tyler_acs1 = []

for year in tyler_estimates_acs1['year'].unique():
    row = tyler_estimates_acs1[tyler_estimates_acs1['year'] == year].iloc[0]
    
    # have to use this formula to calculate total MOE 
    total_moe = np.sqrt(
        row["moved_from_diff_county_same_state_moe"]**2 +
        row["moved_from_diff_state_moe"]**2 +
        row["moved_from_abroad_moe"]**2
    )
    
    migration_data_tyler_acs1.append({
        'year': year,
        'total_in_migration': row['total_in_migration'],
        'moe': total_moe,
        'lower_bound': row['total_in_migration'] - total_moe,
        'upper_bound': row['total_in_migration'] + total_moe
    })

tyler_migration_moes_acs1 = pd.DataFrame(migration_data_tyler_acs1)
tyler_migration_moes_acs1

,year,total_in_migration,moe,lower_bound,upper_bound
0,2005,16814.0,2996.193752,13817.806248,19810.193752
1,2006,13726.0,2707.226810,11018.773190,16433.226810
2,2007,14573.0,2730.362613,11842.637387,17303.362613
3,2008,14059.0,3383.052616,10675.947384,17442.052616
4,2009,15956.0,3405.246834,12550.753166,19361.246834
5,2010,14361.0,3657.367496,10703.632504,18018.367496
6,2011,19403.0,4321.051955,15081.948045,23724.051955
7,2012,20525.0,4612.667233,15912.332767,25137.667233
8,2013,15526.0,3059.751297,12466.248703,18585.751297
9,2014,12223.0,2709.462124,9513.537876,14932.462124


## ACS 5-year

In [27]:
# filter for metro areas in texas only 
msa_tx_acs5 = df_msa_acs5[df_msa_acs5['name'].str.contains('TX', case=False, na=False)].copy()

# some msa_codes have multiple names
# fix that by aggregating instances where there are multiple rows for the same metro area for the same year
# pick just the most common name
msa_tx_acs5['name'] = msa_tx_acs5.groupby('msa_code')['name'].transform(lambda x: x.mode()[0])

# create column for all people who moved to the area from a different location
msa_tx_acs5['total_in_migration'] = (
    msa_tx_acs5['moved_from_diff_county_same_state'] + 
    msa_tx_acs5['moved_from_diff_state'] + 
    msa_tx_acs5['moved_from_abroad']
)

# aggregate to ensure one row per msa_code per year
msa_tx_agg_acs5 = msa_tx_acs5.groupby(['msa_code', 'name', 'year']).agg({
    'population': 'sum',
    'total_families': 'sum',
    'median_age': 'median',
    'population_under_18': 'sum',
    'family_households_with_child': 'sum'
}).reset_index()

print(msa_tx_acs5['msa_code'].nunique())

73


In [28]:
# use function to create multiple dfs at once, pivoting from original so each column is a year
msa_pivots_acs5 = create_pivot_tables(
    df=msa_tx_agg_acs5,
    index_cols=['msa_code','name'],
    variable_names=['population', 
                    'median_age', 
                    'total_families', 
                    'population_under_18', 
                    'family_households_with_child'])

# extract individual pivot tables
msa_tx_pop_acs5 = msa_pivots_acs5['population']
msa_tx_age_acs5 = msa_pivots_acs5['median_age']
msa_tx_families_acs5 = msa_pivots_acs5['total_families']
msa_tx_under18_acs5 = msa_pivots_acs5['population_under_18']
msa_tx_family_households_with_child_acs5 = msa_pivots_acs5['family_households_with_child']

# create df specifically for migration
# this one is a little different because there are multiple values 
msa_tx_migration_acs5 = msa_tx_acs5.pivot(
    index=['msa_code','name'],
    columns='year',
    values=['lived_in_same_house', 
            'moved_within_county', 
            'moved_from_diff_county_same_state',
            'moved_from_diff_state',
            'moved_from_abroad',
            'total_in_migration']
).reset_index()

# flatten cols so theyre not stacked
msa_tx_migration_acs5.columns = [
    f"{var}_{year}" for var, year in msa_tx_migration_acs5.columns
]

# reset index
msa_tx_migration_acs5 = msa_tx_migration_acs5.reset_index();

In [29]:
msa_tx_acs5['year'].unique()

array([2009, 2014, 2019, 2024])

### Population change (ACS5)

In [30]:
# ACS 5-year population change - only non-overlapping periods are valid
msa_tx_pop_acs5['pct_change_19_24'] = ((msa_tx_pop_acs5[2024] - msa_tx_pop_acs5[2019]) / msa_tx_pop_acs5[2019]) * 100 # 5 years
msa_tx_pop_acs5['pct_change_14_24'] = ((msa_tx_pop_acs5[2024] - msa_tx_pop_acs5[2014]) / msa_tx_pop_acs5[2014]) * 100 # 10 years
msa_tx_pop_acs5['pct_change_09_24'] = ((msa_tx_pop_acs5[2024] - msa_tx_pop_acs5[2009]) / msa_tx_pop_acs5[2009]) * 100 # 15 years

# print ranked results
print_rankings(
    df=msa_tx_pop_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_09_24
0                       Mount Pleasant, TX Micro Area         93.181272
1                    Austin-Round Rock, TX Metro Area         52.674134
2                             Longview, TX Metro Area         43.385555
3                              Midland, TX Metro Area         42.547531
4                              Andrews, TX Micro Area         39.977435
5                College Station-Bryan, TX Metro Area         36.541801
6            San Antonio-New Braunfels, TX Metro Area         34.502795
7                       Killeen-Temple, TX Metro Area         34.088456
8                              Lubbock, TX Metro Area         33.201632
9     Houston-The Woodlands-Sugar Land, TX Metro Area         33.019473
10                                Waco, TX Metro Area         32.261822
11         Dallas-Fort Worth-Arlington, TX Metro Area         29.968846
12                              Odessa, TX Metro Area         27

{'pct_change_09_24': year                              name  pct_change_09_24
 0        Mount Pleasant, TX Micro Area         93.181272
 1     Austin-Round Rock, TX Metro Area         52.674134
 2              Longview, TX Metro Area         43.385555
 3               Midland, TX Metro Area         42.547531
 4               Andrews, TX Micro Area         39.977435
 ..                                 ...               ...
 68                Pecos, TX Micro Area               NaN
 69          Port Lavaca, TX Micro Area               NaN
 70             Rockport, TX Micro Area               NaN
 71        Town of Pecos, TX Micro Area               NaN
 72               Zapata, TX Micro Area               NaN
 
 [73 rows x 2 columns],
 'pct_change_14_24': year                              name  pct_change_14_24
 0        Mount Pleasant, TX Micro Area         72.824216
 1              Longview, TX Metro Area         34.278833
 2     Austin-Round Rock, TX Metro Area         32.238193
 3    

### Total number of families (ACS5)

In [31]:
# ACS 5-year families change - only non-overlapping periods are valid
msa_tx_families_acs5['pct_change_19_24'] = ((msa_tx_families_acs5[2024] - msa_tx_families_acs5[2019]) / msa_tx_families_acs5[2019]) * 100 # 5 years
msa_tx_families_acs5['pct_change_14_24'] = ((msa_tx_families_acs5[2024] - msa_tx_families_acs5[2014]) / msa_tx_families_acs5[2014]) * 100 # 10 years
msa_tx_families_acs5['pct_change_09_24'] = ((msa_tx_families_acs5[2024] - msa_tx_families_acs5[2009]) / msa_tx_families_acs5[2009]) * 100 # 15 years

# print ranked results
print_rankings(
    df=msa_tx_families_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_09_24
0                       Mount Pleasant, TX Micro Area         89.136381
1                    Austin-Round Rock, TX Metro Area         63.304302
2                             Longview, TX Metro Area         40.776027
3                              Midland, TX Metro Area         40.494172
4            San Antonio-New Braunfels, TX Metro Area         38.397394
5                College Station-Bryan, TX Metro Area         37.811315
6                       Killeen-Temple, TX Metro Area         36.497293
7     Houston-The Woodlands-Sugar Land, TX Metro Area         36.489430
8                         Stephenville, TX Micro Area         36.475153
9                                 Waco, TX Metro Area         35.767456
10         Dallas-Fort Worth-Arlington, TX Metro Area         33.645912
11                              Odessa, TX Metro Area         32.289718
12                             Lubbock, TX Metro Area         28

{'pct_change_09_24': year                                      name  pct_change_09_24
 0                Mount Pleasant, TX Micro Area         89.136381
 1             Austin-Round Rock, TX Metro Area         63.304302
 2                      Longview, TX Metro Area         40.776027
 3                       Midland, TX Metro Area         40.494172
 4     San Antonio-New Braunfels, TX Metro Area         38.397394
 ..                                         ...               ...
 68                        Pecos, TX Micro Area               NaN
 69                  Port Lavaca, TX Micro Area               NaN
 70                     Rockport, TX Micro Area               NaN
 71                Town of Pecos, TX Micro Area               NaN
 72                       Zapata, TX Micro Area               NaN
 
 [73 rows x 2 columns],
 'pct_change_14_24': year                              name  pct_change_14_24
 0        Mount Pleasant, TX Micro Area         84.451220
 1     Austin-Round Rock, 

### Median age (ACS5)

In [32]:
# ACS 5-year median age change - only non-overlapping periods are valid
msa_tx_age_acs5['change_19_24'] = (msa_tx_age_acs5[2024] - msa_tx_age_acs5[2019])  # 5 years
msa_tx_age_acs5['change_14_24'] = (msa_tx_age_acs5[2024] - msa_tx_age_acs5[2014])  # 10 years
msa_tx_age_acs5['change_09_24'] = (msa_tx_age_acs5[2024] - msa_tx_age_acs5[2009])  # 15 years

# rank by cities that got the youngest
print("\nMetros by median age change (2009-2024) - ACS5:")
print(msa_tx_age_acs5[msa_tx_age_acs5['change_09_24'].notna()]
      .sort_values(by='change_09_24', ascending=True)
      .reset_index(drop=True)[['name', 'change_09_24', 2009, 2024]])


Metros by median age change (2009-2024) - ACS5:
year                                  name  change_09_24  2009  2024
0                     Pampa, TX Micro Area          -2.1  39.1  37.0
1                   Andrews, TX Micro Area          -1.1  34.1  33.0
2                   Midland, TX Metro Area          -0.5  33.4  32.9
3                   Del Rio, TX Micro Area          -0.1  32.4  32.3
4                Big Spring, TX Micro Area          -0.1  36.8  36.7
..                                     ...           ...   ...   ...
59        Austin-Round Rock, TX Metro Area           4.0  32.0  36.0
60              Nacogdoches, TX Micro Area           4.2  27.7  31.9
61    College Station-Bryan, TX Metro Area           4.6  24.0  28.6
62                   Vernon, TX Micro Area           4.6  35.9  40.5
63           Mount Pleasant, TX Micro Area           5.3  32.1  37.4

[64 rows x 4 columns]


### Total population under 18 (ACS5)

In [33]:
# ACS 5-year population under 18 change - only non-overlapping periods are valid
msa_tx_under18_acs5['pct_change_19_24'] = ((msa_tx_under18_acs5[2024] - msa_tx_under18_acs5[2019]) / msa_tx_under18_acs5[2019]) * 100 # 5 years
msa_tx_under18_acs5['pct_change_14_24'] = ((msa_tx_under18_acs5[2024] - msa_tx_under18_acs5[2014]) / msa_tx_under18_acs5[2014]) * 100 # 10 years
msa_tx_under18_acs5['pct_change_09_24'] = ((msa_tx_under18_acs5[2024] - msa_tx_under18_acs5[2009]) / msa_tx_under18_acs5[2009]) * 100 # 15 years

# print ranked results
print_rankings(
    df=msa_tx_under18_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_09_24
0                       Mount Pleasant, TX Micro Area         64.973557
1                              Midland, TX Metro Area         47.429775
2                              Andrews, TX Micro Area         43.889028
3                             Longview, TX Metro Area         36.023005
4                    Austin-Round Rock, TX Metro Area         32.180747
5                               Odessa, TX Metro Area         29.677737
6                              Lubbock, TX Metro Area         27.487562
7                College Station-Bryan, TX Metro Area         25.731858
8                       Killeen-Temple, TX Metro Area         25.264708
9                                 Waco, TX Metro Area         20.957199
10    Houston-The Woodlands-Sugar Land, TX Metro Area         20.275111
11                               Paris, TX Micro Area         19.073726
12                     Sherman-Denison, TX Metro Area         18

{'pct_change_09_24': year                              name  pct_change_09_24
 0        Mount Pleasant, TX Micro Area         64.973557
 1               Midland, TX Metro Area         47.429775
 2               Andrews, TX Micro Area         43.889028
 3              Longview, TX Metro Area         36.023005
 4     Austin-Round Rock, TX Metro Area         32.180747
 ..                                 ...               ...
 68                Pecos, TX Micro Area               NaN
 69          Port Lavaca, TX Micro Area               NaN
 70             Rockport, TX Micro Area               NaN
 71        Town of Pecos, TX Micro Area               NaN
 72               Zapata, TX Micro Area               NaN
 
 [73 rows x 2 columns],
 'pct_change_14_24': year                           name  pct_change_14_24
 0     Mount Pleasant, TX Micro Area         55.128471
 1           Longview, TX Metro Area         33.016312
 2            Midland, TX Metro Area         25.830117
 3              Pa

### Number of family households with children under 18 (ACS5)

In [34]:
# ACS 5-year family households with children change - only non-overlapping periods are valid
msa_tx_family_households_with_child_acs5['pct_change_19_24'] = ((msa_tx_family_households_with_child_acs5[2024] - msa_tx_family_households_with_child_acs5[2019]) / msa_tx_family_households_with_child_acs5[2019]) * 100 # 5 years
msa_tx_family_households_with_child_acs5['pct_change_14_24'] = ((msa_tx_family_households_with_child_acs5[2024] - msa_tx_family_households_with_child_acs5[2014]) / msa_tx_family_households_with_child_acs5[2014]) * 100 # 10 years
msa_tx_family_households_with_child_acs5['pct_change_09_24'] = ((msa_tx_family_households_with_child_acs5[2024] - msa_tx_family_households_with_child_acs5[2009]) / msa_tx_family_households_with_child_acs5[2009]) * 100 # 15 years

# print ranked results
print_rankings(
    df=msa_tx_family_households_with_child_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                                             name  pct_change_09_24
0                       Mount Pleasant, TX Micro Area         64.870932
1                              Midland, TX Metro Area         50.217085
2                    Austin-Round Rock, TX Metro Area         45.012852
3                              Andrews, TX Micro Area         44.867886
4                               Odessa, TX Metro Area         38.185095
5                             Longview, TX Metro Area         36.012646
6                         Stephenville, TX Micro Area         33.699835
7                College Station-Bryan, TX Metro Area         27.649065
8            San Antonio-New Braunfels, TX Metro Area         26.907925
9                       Fredericksburg, TX Micro Area         25.190840
10                      Killeen-Temple, TX Metro Area         24.316198
11    Houston-The Woodlands-Sugar Land, TX Metro Area         23.299162
12                                Waco, TX Metro Area         22

{'pct_change_09_24': year                              name  pct_change_09_24
 0        Mount Pleasant, TX Micro Area         64.870932
 1               Midland, TX Metro Area         50.217085
 2     Austin-Round Rock, TX Metro Area         45.012852
 3               Andrews, TX Micro Area         44.867886
 4                Odessa, TX Metro Area         38.185095
 ..                                 ...               ...
 68                Pecos, TX Micro Area               NaN
 69          Port Lavaca, TX Micro Area               NaN
 70             Rockport, TX Micro Area               NaN
 71        Town of Pecos, TX Micro Area               NaN
 72               Zapata, TX Micro Area               NaN
 
 [73 rows x 2 columns],
 'pct_change_14_24': year                              name  pct_change_14_24
 0        Mount Pleasant, TX Micro Area         84.269945
 1              Longview, TX Metro Area         38.118980
 2               Midland, TX Metro Area         36.524613
 3    

### Migration to Texas MSAs (ACS5)

In [35]:
# ACS 5-year migration change - only non-overlapping periods are valid
msa_tx_migration_acs5['pct_chg_migration_09_24'] = ((msa_tx_migration_acs5['total_in_migration_2024'] - msa_tx_migration_acs5['total_in_migration_2009']) / msa_tx_migration_acs5['total_in_migration_2009'] * 100)
msa_tx_migration_acs5['pct_chg_migration_14_24'] = ((msa_tx_migration_acs5['total_in_migration_2024'] - msa_tx_migration_acs5['total_in_migration_2014']) / msa_tx_migration_acs5['total_in_migration_2014'] * 100)
msa_tx_migration_acs5['pct_chg_migration_19_24'] = ((msa_tx_migration_acs5['total_in_migration_2024'] - msa_tx_migration_acs5['total_in_migration_2019']) / msa_tx_migration_acs5['total_in_migration_2019'] * 100)

# Print ranked results
print_rankings(
    df=msa_tx_migration_acs5,
    name_col='name_',
    variable_names=['pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

                                              name_  pct_chg_migration_09_24
0                            Andrews, TX Micro Area                80.529595
1                       Jacksonville, TX Micro Area                76.741071
2                            Midland, TX Metro Area                65.308151
3                    Sherman-Denison, TX Metro Area                58.253558
4                       Stephenville, TX Micro Area                53.927987
5                             Odessa, TX Metro Area                47.248085
6                            Brenham, TX Micro Area                46.729958
7                           El Campo, TX Micro Area                36.303030
8                  Austin-Round Rock, TX Metro Area                35.685288
9                            Lubbock, TX Metro Area                35.356608
10                    Mount Pleasant, TX Micro Area                29.406221
11                             Paris, TX Micro Area                29.366736

{'pct_chg_migration_09_24':                              name_  pct_chg_migration_09_24
 0           Andrews, TX Micro Area                80.529595
 1      Jacksonville, TX Micro Area                76.741071
 2           Midland, TX Metro Area                65.308151
 3   Sherman-Denison, TX Metro Area                58.253558
 4      Stephenville, TX Micro Area                53.927987
 ..                             ...                      ...
 68            Pecos, TX Micro Area                      NaN
 69      Port Lavaca, TX Micro Area                      NaN
 70         Rockport, TX Micro Area                      NaN
 71    Town of Pecos, TX Micro Area                      NaN
 72           Zapata, TX Micro Area                      NaN
 
 [73 rows x 2 columns],
 'pct_chg_migration_14_24':                              name_  pct_chg_migration_14_24
 0          El Campo, TX Micro Area               111.372180
 1           Brenham, TX Micro Area               108.754377
 2   

## ACS5 margins of error

In [36]:
tyler_estimates_acs5 = msa_tx_acs5[msa_tx_acs5['name'] == 'Tyler, TX Metro Area'].copy()
tyler_estimates_acs5

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,median_age_moe,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,msa_code,year,total_in_migration
444,"Tyler, TX Metro Area",197631.0,48076.0,23912.0,35.7,51096.0,155721.0,23260.0,10612.0,3891.0,...,0.1,0.0,2186.0,1915.0,1195.0,848.0,322.0,46340,2009,15296.0
1607,"Tyler, TX Metro Area",214735.0,53927.0,26596.0,36.0,54296.0,172408.0,23577.0,10885.0,4206.0,...,0.2,<NA>,1791.0,1568.0,1188.0,762.0,304.0,46340,2014,15839.0
2549,"Tyler, TX Metro Area",227449.0,53919.0,25385.0,36.6,56012.0,195995.0,14368.0,10144.0,3540.0,...,0.2,<NA>,2119.0,1645.0,1281.0,809.0,352.0,46340,2019,14649.0
3679,"Tyler, TX Metro Area",241740.0,57522.0,27097.0,37.5,58485.0,210107.0,14964.0,9982.0,3187.0,...,0.3,<NA>,2237.0,1753.0,1160.0,737.0,222.0,46340,2024,13739.0


In [37]:
# Create migration confidence intervals dataframe for Smith County
migration_data_tyler_acs5 = []

for year in tyler_estimates_acs5['year'].unique():
    row = tyler_estimates_acs5[tyler_estimates_acs5['year'] == year].iloc[0]
    
    # Calculate total MOE
    total_moe = np.sqrt(
        row["moved_from_diff_county_same_state_moe"]**2 +
        row["moved_from_diff_state_moe"]**2 +
        row["moved_from_abroad_moe"]**2
    )
    
    migration_data_tyler_acs5.append({
        'year': year,
        'total_in_migration': row['total_in_migration'],
        'moe': total_moe,
        'lower_bound': row['total_in_migration'] - total_moe,
        'upper_bound': row['total_in_migration'] + total_moe
    })

tyler_migration_moes_acs5 = pd.DataFrame(migration_data_tyler_acs5)
tyler_migration_moes_acs5

,year,total_in_migration,moe,lower_bound,upper_bound
0,2009,15296.0,1500.270976,13795.729024,16796.270976
1,2014,15839.0,1443.746515,14395.253485,17282.746515
2,2019,14649.0,1555.424701,13093.575299,16204.424701
3,2024,13739.0,1392.139720,12346.860280,15131.139720


# County-level analysis

## ACS 1-year

In [38]:
# We already created tx_counties
# Use helper function to create multiple pivot tables at once
county_pivots = create_pivot_tables(
    df=tx_counties,
    index_cols=['county_code','name'],
    variable_names=['population', 'median_age', 'total_families', 'population_under_18', 'family_households_with_child']
)

tx_counties['total_in_migration'] = (
    tx_counties['moved_from_diff_county_same_state'] + 
    tx_counties['moved_from_diff_state'] + 
    tx_counties['moved_from_abroad']
)

county_tx_pop = county_pivots['population']
county_tx_age = county_pivots['median_age']
county_tx_families = county_pivots['total_families']
county_tx_under18 = county_pivots['population_under_18']
county_tx_family_households_with_child = county_pivots['family_households_with_child']

tx_counties_migration = tx_counties.pivot(
    index=['county_code','name'],
    columns='year',
    values=['lived_in_same_house', 
            'moved_within_county', 
            'moved_from_diff_county_same_state',
            'moved_from_diff_state',
            'moved_from_abroad',
            'total_in_migration']
) 

# flatten cols so theyre not stacked
tx_counties_migration.columns = [
    f"{var}_{year}" for var, year in tx_counties_migration.columns
]

# reset index
tx_counties_migration = tx_counties_migration.reset_index();

### County Population Change Analysis

In [39]:
# create columns for population change in the past 5, 10, 15 and 19 years (2005–2024)
county_tx_pop['pct_change_19_24'] = ((county_tx_pop[2024] - county_tx_pop[2019]) / county_tx_pop[2019]) * 100 # 5 years
county_tx_pop['pct_change_14_24'] = ((county_tx_pop[2024] - county_tx_pop[2014]) / county_tx_pop[2014]) * 100 # 10 years
county_tx_pop['pct_change_09_24'] = ((county_tx_pop[2024] - county_tx_pop[2009]) / county_tx_pop[2009]) * 100 # 15 years
county_tx_pop['pct_change_05_24'] = ((county_tx_pop[2024] - county_tx_pop[2005]) / county_tx_pop[2005]) * 100 # 19 years

# Print ranked results
print_rankings(
    df=county_tx_pop,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                        name  pct_change_05_24
0             Hays County, Texas        153.872033
1          Kaufman County, Texas        125.690491
2       Williamson County, Texas        121.508369
3            Comal County, Texas        112.701226
4        Fort Bend County, Texas        109.619771
5       Montgomery County, Texas         99.338122
6        Guadalupe County, Texas         92.783200
7           Denton County, Texas         91.937353
8           Collin County, Texas         91.260591
9          Bastrop County, Texas         86.201478
10          Parker County, Texas         79.984176
11           Ellis County, Texas         76.747034
12          Brazos County, Texas         73.951583
13         Liberty County, Texas         64.073821
14            Bell County, Texas         62.040131
15          Travis County, Texas         57.415430
16        Brazoria County, Texas         54.547903
17         Midland County, Texas         53.346976
18            Hunt County, Texa

{'pct_change_05_24': year                        name  pct_change_05_24
 0             Hays County, Texas        153.872033
 1          Kaufman County, Texas        125.690491
 2       Williamson County, Texas        121.508369
 3            Comal County, Texas        112.701226
 4        Fort Bend County, Texas        109.619771
 5       Montgomery County, Texas         99.338122
 6        Guadalupe County, Texas         92.783200
 7           Denton County, Texas         91.937353
 8           Collin County, Texas         91.260591
 9          Bastrop County, Texas         86.201478
 10          Parker County, Texas         79.984176
 11           Ellis County, Texas         76.747034
 12          Brazos County, Texas         73.951583
 13         Liberty County, Texas         64.073821
 14            Bell County, Texas         62.040131
 15          Travis County, Texas         57.415430
 16        Brazoria County, Texas         54.547903
 17         Midland County, Texas         53

### Number of families

In [40]:
# create columns for percentage change in the number of families
county_tx_families['pct_change_19_24'] = ((county_tx_families[2024] - county_tx_families[2019]) / county_tx_families[2019]) * 100 # 5 years
county_tx_families['pct_change_14_24'] = ((county_tx_families[2024] - county_tx_families[2014]) / county_tx_families[2014]) * 100 # 10 years
county_tx_families['pct_change_09_24'] = ((county_tx_families[2024] - county_tx_families[2009]) / county_tx_families[2009]) * 100 # 15 years
county_tx_families['pct_change_05_24'] = ((county_tx_families[2024] - county_tx_families[2005]) / county_tx_families[2005]) * 100 # 19 years

# Print ranked results
print_rankings(
    df=county_tx_families,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                        name  pct_change_05_24
0             Hays County, Texas        159.685109
1       Williamson County, Texas        132.381552
2        Fort Bend County, Texas        131.091559
3            Comal County, Texas        115.406545
4           Denton County, Texas        110.410813
5       Montgomery County, Texas        104.294636
6           Collin County, Texas         96.455155
7          Bastrop County, Texas         94.159602
8        Guadalupe County, Texas         87.060608
9            Ellis County, Texas         81.301542
10         Kaufman County, Texas         75.400483
11          Brazos County, Texas         73.852683
12          Travis County, Texas         65.761376
13          Parker County, Texas         59.610080
14         Midland County, Texas         58.895340
15            Bell County, Texas         52.034514
16            Hunt County, Texas         50.854432
17         Liberty County, Texas         50.777810
18         Johnson County, Texa

{'pct_change_05_24': year                        name  pct_change_05_24
 0             Hays County, Texas        159.685109
 1       Williamson County, Texas        132.381552
 2        Fort Bend County, Texas        131.091559
 3            Comal County, Texas        115.406545
 4           Denton County, Texas        110.410813
 5       Montgomery County, Texas        104.294636
 6           Collin County, Texas         96.455155
 7          Bastrop County, Texas         94.159602
 8        Guadalupe County, Texas         87.060608
 9            Ellis County, Texas         81.301542
 10         Kaufman County, Texas         75.400483
 11          Brazos County, Texas         73.852683
 12          Travis County, Texas         65.761376
 13          Parker County, Texas         59.610080
 14         Midland County, Texas         58.895340
 15            Bell County, Texas         52.034514
 16            Hunt County, Texas         50.854432
 17         Liberty County, Texas         50

In [41]:
# Export county families data with percentage changes to CSV
county_families_export = county_tx_families[['county_code', 'name', 'pct_change_05_24', 'pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']].copy()
county_families_export['name'] = county_families_export['name'].str.replace(' County, Texas', '', regex=False)
county_families_export.to_csv('county_families_pct_changes.csv', index=False)
print("Exported to county_families_pct_changes.csv")

Exported to county_families_pct_changes.csv


### Family households with children under 18

In [42]:
# cols for percentage change in the number of families with at least one kid
county_tx_family_households_with_child['pct_change_19_24'] = ((county_tx_family_households_with_child[2024] - county_tx_family_households_with_child[2019]) / county_tx_family_households_with_child[2019]) * 100 # 5 years
county_tx_family_households_with_child['pct_change_14_24'] = ((county_tx_family_households_with_child[2024] - county_tx_family_households_with_child[2014]) / county_tx_family_households_with_child[2014]) * 100 # 10 years
county_tx_family_households_with_child['pct_change_09_24'] = ((county_tx_family_households_with_child[2024] - county_tx_family_households_with_child[2009]) / county_tx_family_households_with_child[2009]) * 100 # 15 years
county_tx_family_households_with_child['pct_change_05_24'] = ((county_tx_family_households_with_child[2024] - county_tx_family_households_with_child[2005]) / county_tx_family_households_with_child[2005]) * 100 # 19 years

# Print ranked results
print_rankings(
    df=county_tx_family_households_with_child,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                        name  pct_change_05_24
0             Hays County, Texas        151.499036
1       Williamson County, Texas        108.450401
2            Comal County, Texas        103.687858
3        Fort Bend County, Texas        100.633578
4        Guadalupe County, Texas         96.395672
5       Montgomery County, Texas         90.183548
6          Bastrop County, Texas         83.163642
7           Collin County, Texas         79.391997
8          Liberty County, Texas         75.860038
9           Denton County, Texas         74.477575
10         Kaufman County, Texas         70.799385
11          Brazos County, Texas         58.810600
12            Hunt County, Texas         54.573658
13         Midland County, Texas         53.415702
14           Ellis County, Texas         51.729156
15          Parker County, Texas         51.271836
16            Bell County, Texas         40.845217
17          Travis County, Texas         37.021448
18           Ector County, Texa

{'pct_change_05_24': year                        name  pct_change_05_24
 0             Hays County, Texas        151.499036
 1       Williamson County, Texas        108.450401
 2            Comal County, Texas        103.687858
 3        Fort Bend County, Texas        100.633578
 4        Guadalupe County, Texas         96.395672
 5       Montgomery County, Texas         90.183548
 6          Bastrop County, Texas         83.163642
 7           Collin County, Texas         79.391997
 8          Liberty County, Texas         75.860038
 9           Denton County, Texas         74.477575
 10         Kaufman County, Texas         70.799385
 11          Brazos County, Texas         58.810600
 12            Hunt County, Texas         54.573658
 13         Midland County, Texas         53.415702
 14           Ellis County, Texas         51.729156
 15          Parker County, Texas         51.271836
 16            Bell County, Texas         40.845217
 17          Travis County, Texas         37

### Total people under 18

In [43]:
# create columns for population change for people under 18 
county_tx_under18['pct_change_19_24'] = ((county_tx_under18[2024] - county_tx_under18[2019]) / county_tx_under18[2019]) * 100 # 5 years
county_tx_under18['pct_change_14_24'] = ((county_tx_under18[2024] - county_tx_under18[2014]) / county_tx_under18[2014]) * 100 # 10 years
county_tx_under18['pct_change_09_24'] = ((county_tx_under18[2024] - county_tx_under18[2009]) / county_tx_under18[2009]) * 100 # 15 years
county_tx_under18['pct_change_05_24'] = ((county_tx_under18[2024] - county_tx_under18[2005]) / county_tx_under18[2005]) * 100 # 19 years

# Print ranked results
print_rankings(
    df=county_tx_under18,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

year                        name  pct_change_05_24
0          Kaufman County, Texas        139.352906
1             Hays County, Texas        130.239327
2        Fort Bend County, Texas         95.064697
3       Montgomery County, Texas         86.812906
4       Williamson County, Texas         81.817695
5        Guadalupe County, Texas         74.669728
6           Parker County, Texas         73.329630
7           Collin County, Texas         66.061170
8            Ellis County, Texas         65.072005
9          Midland County, Texas         60.094493
10          Denton County, Texas         58.946922
11          Brazos County, Texas         52.918013
12            Hunt County, Texas         43.167867
13           Ector County, Texas         38.516101
14         Johnson County, Texas         36.922998
15        Brazoria County, Texas         35.194375
16            Bell County, Texas         34.649680
17         Randall County, Texas         30.601994
18         Grayson County, Texa

{'pct_change_05_24': year                        name  pct_change_05_24
 0          Kaufman County, Texas        139.352906
 1             Hays County, Texas        130.239327
 2        Fort Bend County, Texas         95.064697
 3       Montgomery County, Texas         86.812906
 4       Williamson County, Texas         81.817695
 5        Guadalupe County, Texas         74.669728
 6           Parker County, Texas         73.329630
 7           Collin County, Texas         66.061170
 8            Ellis County, Texas         65.072005
 9          Midland County, Texas         60.094493
 10          Denton County, Texas         58.946922
 11          Brazos County, Texas         52.918013
 12            Hunt County, Texas         43.167867
 13           Ector County, Texas         38.516101
 14         Johnson County, Texas         36.922998
 15        Brazoria County, Texas         35.194375
 16            Bell County, Texas         34.649680
 17         Randall County, Texas         30

### County Migration Analysis

In [44]:
# migration
tx_counties_migration['pct_chg_migration_05_24'] = ((tx_counties_migration['total_in_migration_2024'] - tx_counties_migration['total_in_migration_2005']) / tx_counties_migration['total_in_migration_2005'] * 100)
tx_counties_migration['pct_chg_migration_09_24'] = ((tx_counties_migration['total_in_migration_2024'] - tx_counties_migration['total_in_migration_2009']) / tx_counties_migration['total_in_migration_2009'] * 100)
tx_counties_migration['pct_chg_migration_14_24'] = ((tx_counties_migration['total_in_migration_2024'] - tx_counties_migration['total_in_migration_2014']) / tx_counties_migration['total_in_migration_2014'] * 100)
tx_counties_migration['pct_chg_migration_19_24'] = ((tx_counties_migration['total_in_migration_2024'] - tx_counties_migration['total_in_migration_2019']) / tx_counties_migration['total_in_migration_2019'] * 100)

# Print ranked results
print_rankings(
    df=tx_counties_migration,
    variable_names=['pct_chg_migration_05_24', 'pct_chg_migration_09_24',
                 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

                          name  pct_chg_migration_05_24
0        Midland County, Texas               248.397234
1           Hunt County, Texas               232.508105
2        Bastrop County, Texas               230.826307
3        Kaufman County, Texas               226.609724
4          Ector County, Texas               199.605263
5           Hays County, Texas               124.803569
6         Brazos County, Texas                93.648208
7     Williamson County, Texas                85.874606
8        Lubbock County, Texas                80.895458
9      Guadalupe County, Texas                62.192864
10        Potter County, Texas                56.850575
11       Randall County, Texas                55.568401
12         Bexar County, Texas                52.315018
13         Comal County, Texas                51.468564
14       El Paso County, Texas                50.652884
15       Johnson County, Texas                50.633302
16        Denton County, Texas                48

{'pct_chg_migration_05_24':                           name  pct_chg_migration_05_24
 0        Midland County, Texas               248.397234
 1           Hunt County, Texas               232.508105
 2        Bastrop County, Texas               230.826307
 3        Kaufman County, Texas               226.609724
 4          Ector County, Texas               199.605263
 5           Hays County, Texas               124.803569
 6         Brazos County, Texas                93.648208
 7     Williamson County, Texas                85.874606
 8        Lubbock County, Texas                80.895458
 9      Guadalupe County, Texas                62.192864
 10        Potter County, Texas                56.850575
 11       Randall County, Texas                55.568401
 12         Bexar County, Texas                52.315018
 13         Comal County, Texas                51.468564
 14       El Paso County, Texas                50.652884
 15       Johnson County, Texas                50.633302
 16 

## ACS1 margins of error

In [45]:
smith_estimates = tx_counties[tx_counties['name'] == 'Smith County, Texas'].copy()
smith_estimates

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,state,county_code,year,total_in_migration
40,"Smith County, Texas",185465.0,48152.0,26026.0,35.4,49255.0,144741.0,20038.0,12408.0,3638.0,...,177.0,4081.0,3858.0,2456.0,1629.0,540.0,48,423,2005,16814.0
90,"Smith County, Texas",194635.0,48471.0,24303.0,35.2,49980.0,152096.0,24363.0,7647.0,4786.0,...,279.0,5212.0,4351.0,1975.0,1536.0,1034.0,48,423,2006,13726.0
122,"Smith County, Texas",198705.0,48396.0,24532.0,35.3,50624.0,154608.0,25920.0,10017.0,4053.0,...,251.0,5101.0,4337.0,2316.0,1380.0,432.0,48,423,2007,14573.0
172,"Smith County, Texas",201277.0,48756.0,24358.0,35.8,50898.0,161619.0,23302.0,10093.0,3134.0,...,329.0,5316.0,4955.0,2644.0,1990.0,703.0,48,423,2008,14059.0
225,"Smith County, Texas",204665.0,47517.0,21899.0,36.1,52773.0,165461.0,20022.0,11575.0,3633.0,...,0.0,4666.0,4029.0,3036.0,1409.0,627.0,48,423,2009,15956.0
291,"Smith County, Texas",210512.0,53601.0,26382.0,35.5,54236.0,166384.0,26636.0,10171.0,2857.0,...,524.0,5540.0,5242.0,2945.0,1916.0,1016.0,48,423,2010,14361.0
344,"Smith County, Texas",213381.0,53407.0,25992.0,35.6,54404.0,165910.0,25906.0,12872.0,6230.0,...,292.0,6348.0,4968.0,3165.0,2931.0,252.0,48,423,2011,19403.0
365,"Smith County, Texas",214821.0,54091.0,26429.0,36.0,54602.0,170460.0,21024.0,14251.0,5551.0,...,293.0,6012.0,4227.0,4031.0,2173.0,553.0,48,423,2012,20525.0
418,"Smith County, Texas",216080.0,53610.0,27734.0,36.2,54209.0,174730.0,22026.0,9675.0,5403.0,...,<NA>,6449.0,4986.0,2275.0,2022.0,313.0,48,423,2013,15526.0
470,"Smith County, Texas",218842.0,53239.0,25000.0,36.3,54587.0,178721.0,24562.0,8009.0,3510.0,...,57.0,5087.0,4560.0,2168.0,1540.0,519.0,48,423,2014,12223.0


In [46]:
# Create migration confidence intervals dataframe
migration_data_acs1 = []

for year in smith_estimates['year'].unique():
    row = smith_estimates[smith_estimates['year'] == year].iloc[0]
    
    # Calculate total MOE by summing component MOEs
    total_moe = np.sqrt(
        row["moved_from_diff_county_same_state_moe"]**2 +
        row["moved_from_diff_state_moe"]**2 +
        row["moved_from_abroad_moe"]**2
    )
    
    migration_data_acs1.append({
        'year': year,
        'total_in_migration': row['total_in_migration'],
        'moe': total_moe,
        'lower_bound': row['total_in_migration'] - total_moe,
        'upper_bound': row['total_in_migration'] + total_moe
    })

smith_migration_acs1 = pd.DataFrame(migration_data_acs1)
smith_migration_acs1

,year,total_in_migration,moe,lower_bound,upper_bound
0,2005,16814.0,2996.193752,13817.806248,19810.193752
1,2006,13726.0,2707.226810,11018.773190,16433.226810
2,2007,14573.0,2730.362613,11842.637387,17303.362613
3,2008,14059.0,3383.052616,10675.947384,17442.052616
4,2009,15956.0,3405.246834,12550.753166,19361.246834
5,2010,14361.0,3657.367496,10703.632504,18018.367496
6,2011,19403.0,4321.051955,15081.948045,23724.051955
7,2012,20525.0,4612.667233,15912.332767,25137.667233
8,2013,15526.0,3059.751297,12466.248703,18585.751297
9,2014,12223.0,2709.462124,9513.537876,14932.462124


## ACS 5-year analysis

In [47]:
# Use helper function to create multiple pivot tables at once

# check out the data
county_pivots_acs5 = create_pivot_tables(
    df=tx_counties_acs5,
    index_cols=['county_code','name'],
    variable_names=['population', 'median_age', 'total_families', 'population_under_18', 'family_households_with_child']
)

tx_counties_acs5['total_in_migration'] = (
    tx_counties_acs5['moved_from_diff_county_same_state'] + 
    tx_counties_acs5['moved_from_diff_state'] + 
    tx_counties_acs5['moved_from_abroad']
)

county_tx_pop_acs5 = county_pivots_acs5['population']
county_tx_age_acs5 = county_pivots_acs5['median_age']
county_tx_families_acs5 = county_pivots_acs5['total_families']
county_tx_under18_acs5 = county_pivots_acs5['population_under_18']
county_tx_family_households_with_child_acs5 = county_pivots_acs5['family_households_with_child']

tx_counties_migration_acs5 = tx_counties_acs5.pivot(
    index=['county_code','name'],
    columns='year',
    values=['lived_in_same_house', 
            'moved_within_county', 
            'moved_from_diff_county_same_state',
            'moved_from_diff_state',
            'moved_from_abroad',
            'total_in_migration']
) 

# flatten cols so theyre not stacked
tx_counties_migration_acs5.columns = [
    f"{var}_{year}" for var, year in tx_counties_migration_acs5.columns
]

# reset index
tx_counties_migration_acs5 = tx_counties_migration_acs5.reset_index();

### Population change (ACS5)

In [48]:
county_tx_pop_acs5

year,county_code,name,2009,2014,2019,2024
0,001,"Anderson County, Texas",56575,58084,57810,58439
1,003,"Andrews County, Texas",13295,16126,18036,18610
2,005,"Angelina County, Texas",82465,87433,87322,87275
3,007,"Aransas County, Texas",24499,23889,24462,24876
4,009,"Archer County, Texas",8936,8853,8716,8867
...,...,...,...,...,...,...
249,499,"Wood County, Texas",42104,42431,44366,46961
250,501,"Yoakum County, Texas",7453,8054,8631,7571
251,503,"Young County, Texas",17701,18374,18036,18029
252,505,"Zapata County, Texas",13561,14231,14304,13841


In [49]:
# ACS 5-year population change - only non-overlapping periods are valid
county_tx_pop_acs5['pct_change_19_24'] = ((county_tx_pop_acs5[2024] - county_tx_pop_acs5[2019]) / county_tx_pop_acs5[2019]) * 100 # 5 years
county_tx_pop_acs5['pct_change_14_24'] = ((county_tx_pop_acs5[2024] - county_tx_pop_acs5[2014]) / county_tx_pop_acs5[2014]) * 100 # 10 years
county_tx_pop_acs5['pct_change_09_24'] = ((county_tx_pop_acs5[2024] - county_tx_pop_acs5[2009]) / county_tx_pop_acs5[2009]) * 100 # 15 years

# Print ranked results
print_rankings(
    df=county_tx_pop_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                         name  pct_change_09_24
0              Hays County, Texas         90.023414
1        Williamson County, Texas         80.617445
2           Kaufman County, Texas         79.808944
3          Chambers County, Texas         76.375094
4         Fort Bend County, Texas         75.904105
5             Comal County, Texas         75.139101
6            Waller County, Texas         73.132313
7          Rockwall County, Texas         69.770923
8        Montgomery County, Texas         66.234826
9         Guadalupe County, Texas         63.576118
10           Denton County, Texas         60.100320
11           Collin County, Texas         59.467399
12          Kendall County, Texas         55.195884
13           Parker County, Texas         52.475906
14           Gaines County, Texas         49.811321
15            Ellis County, Texas         49.745694
16          Bastrop County, Texas         48.178734
17           Brazos County, Texas         41.386493
18          

{'pct_change_09_24': year                      name  pct_change_09_24
 0           Hays County, Texas         90.023414
 1     Williamson County, Texas         80.617445
 2        Kaufman County, Texas         79.808944
 3       Chambers County, Texas         76.375094
 4      Fort Bend County, Texas         75.904105
 ..                         ...               ...
 249     Crockett County, Texas        -25.422833
 250      Dickens County, Texas        -29.528035
 251      Edwards County, Texas        -32.389937
 252       Kenedy County, Texas        -56.845238
 253       Loving County, Texas        -59.259259
 
 [254 rows x 2 columns],
 'pct_change_14_24': year                      name  pct_change_14_24
 0        Kaufman County, Texas         61.132946
 1          Comal County, Texas         58.733421
 2           Hays County, Texas         57.642157
 3       Rockwall County, Texas         48.508512
 4     Williamson County, Texas         47.126316
 ..                         ...  

### Number of families (ACS5)

In [50]:
# ACS 5-year families change - only non-overlapping periods are valid
county_tx_families_acs5['pct_change_19_24'] = ((county_tx_families_acs5[2024] - county_tx_families_acs5[2019]) / county_tx_families_acs5[2019]) * 100 # 5 years
county_tx_families_acs5['pct_change_14_24'] = ((county_tx_families_acs5[2024] - county_tx_families_acs5[2014]) / county_tx_families_acs5[2014]) * 100 # 10 years
county_tx_families_acs5['pct_change_09_24'] = ((county_tx_families_acs5[2024] - county_tx_families_acs5[2009]) / county_tx_families_acs5[2009]) * 100 # 15 years

# Print ranked results
print_rankings(
    df=county_tx_families_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                         name  pct_change_09_24
0              Hays County, Texas        123.573773
1         Fort Bend County, Texas        103.856381
2        Williamson County, Texas         91.989684
3            Denton County, Texas         80.286680
4             Comal County, Texas         76.079218
5           Kaufman County, Texas         72.932065
6        Montgomery County, Texas         71.888188
7          Rockwall County, Texas         69.486911
8         Guadalupe County, Texas         67.092263
9           Bastrop County, Texas         64.299124
10           Collin County, Texas         62.928015
11           Parker County, Texas         60.386258
12        Somervell County, Texas         57.314974
13           Wilson County, Texas         53.593363
14            Ellis County, Texas         53.368553
15           Waller County, Texas         52.478483
16           Blanco County, Texas         52.161264
17          Kendall County, Texas         52.019804
18         C

{'pct_change_09_24': year                      name  pct_change_09_24
 0           Hays County, Texas        123.573773
 1      Fort Bend County, Texas        103.856381
 2     Williamson County, Texas         91.989684
 3         Denton County, Texas         80.286680
 4          Comal County, Texas         76.079218
 ..                         ...               ...
 249        Foard County, Texas        -39.320388
 250     Jim Hogg County, Texas        -40.076336
 251      Edwards County, Texas        -46.840959
 252       Kenedy County, Texas        -67.647059
 253       Loving County, Texas        -89.189189
 
 [254 rows x 2 columns],
 'pct_change_14_24': year                      name  pct_change_14_24
 0           Hays County, Texas         69.179150
 1          Comal County, Texas         60.451639
 2     Williamson County, Texas         53.354774
 3     Montgomery County, Texas         47.107750
 4      Fort Bend County, Texas         45.081402
 ..                         ...  

### Family households with children under 18 (ACS5)

In [51]:
# ACS 5-year family households with children change - only non-overlapping periods are valid
county_tx_family_households_with_child_acs5['pct_change_19_24'] = ((county_tx_family_households_with_child_acs5[2024] - county_tx_family_households_with_child_acs5[2019]) / county_tx_family_households_with_child_acs5[2019]) * 100 # 5 years
county_tx_family_households_with_child_acs5['pct_change_14_24'] = ((county_tx_family_households_with_child_acs5[2024] - county_tx_family_households_with_child_acs5[2014]) / county_tx_family_households_with_child_acs5[2014]) * 100 # 10 years
county_tx_family_households_with_child_acs5['pct_change_09_24'] = ((county_tx_family_households_with_child_acs5[2024] - county_tx_family_households_with_child_acs5[2009]) / county_tx_family_households_with_child_acs5[2009]) * 100 # 15 years

# Print ranked results
print_rankings(
    df=county_tx_family_households_with_child_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                         name  pct_change_09_24
0              King County, Texas        177.777778
1              Hays County, Texas        110.952226
2         Somervell County, Texas         86.573427
3           Kaufman County, Texas         84.843162
4         Fort Bend County, Texas         82.699973
5        Williamson County, Texas         77.476498
6         Guadalupe County, Texas         62.432809
7        Montgomery County, Texas         62.424296
8            Denton County, Texas         55.906206
9             Mason County, Texas         55.862069
10         Rockwall County, Texas         55.520737
11            Comal County, Texas         54.314054
12           Parker County, Texas         51.928850
13           Collin County, Texas         49.215962
14          Kendall County, Texas         48.808681
15           Waller County, Texas         47.409326
16          Midland County, Texas         45.723434
17            Rains County, Texas         45.218418
18          

{'pct_change_09_24': year                     name  pct_change_09_24
 0          King County, Texas        177.777778
 1          Hays County, Texas        110.952226
 2     Somervell County, Texas         86.573427
 3       Kaufman County, Texas         84.843162
 4     Fort Bend County, Texas         82.699973
 ..                        ...               ...
 249     Terrell County, Texas        -60.447761
 250        Kent County, Texas        -61.538462
 251       Foard County, Texas        -66.511628
 252      Kenedy County, Texas        -80.851064
 253      Loving County, Texas       -100.000000
 
 [254 rows x 2 columns],
 'pct_change_14_24': year                    name  pct_change_14_24
 0     McMullen County, Texas        151.282051
 1         King County, Texas        117.391304
 2         Hays County, Texas         56.341828
 3       Waller County, Texas         49.585440
 4        Comal County, Texas         49.210155
 ..                       ...               ...
 249     

### Total people under 18 (ACS5)

In [52]:
# ACS 5-year population under 18 change - only non-overlapping periods are valid
county_tx_under18_acs5['pct_change_19_24'] = ((county_tx_under18_acs5[2024] - county_tx_under18_acs5[2019]) / county_tx_under18_acs5[2019]) * 100 # 5 years
county_tx_under18_acs5['pct_change_14_24'] = ((county_tx_under18_acs5[2024] - county_tx_under18_acs5[2014]) / county_tx_under18_acs5[2014]) * 100 # 10 years
county_tx_under18_acs5['pct_change_09_24'] = ((county_tx_under18_acs5[2024] - county_tx_under18_acs5[2009]) / county_tx_under18_acs5[2009]) * 100 # 15 years

# Print ranked results
print_rankings(
    df=county_tx_under18_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

year                         name  pct_change_09_24
0            Karnes County, Texas        134.894992
1            Borden County, Texas        107.438017
2          Sterling County, Texas         81.138790
3           Kaufman County, Texas         78.644934
4              Hays County, Texas         75.548041
5          Chambers County, Texas         74.115617
6            Waller County, Texas         62.604341
7            Gaines County, Texas         62.436238
8         Fort Bend County, Texas         57.813183
9             Comal County, Texas         57.232388
10          Liberty County, Texas         55.024547
11       Montgomery County, Texas         53.667005
12       Williamson County, Texas         53.030446
13         Rockwall County, Texas         52.228471
14        Guadalupe County, Texas         48.119747
15           Parker County, Texas         45.745366
16          Bastrop County, Texas         44.911528
17          Andrews County, Texas         43.889028
18          

{'pct_change_09_24': year                    name  pct_change_09_24
 0       Karnes County, Texas        134.894992
 1       Borden County, Texas        107.438017
 2     Sterling County, Texas         81.138790
 3      Kaufman County, Texas         78.644934
 4         Hays County, Texas         75.548041
 ..                       ...               ...
 249      Foard County, Texas        -54.268293
 250    Sherman County, Texas        -57.598300
 251     Kinney County, Texas        -68.275862
 252     Kenedy County, Texas        -78.494624
 253     Loving County, Texas       -100.000000
 
 [254 rows x 2 columns],
 'pct_change_14_24': year                    name  pct_change_14_24
 0     McMullen County, Texas        224.418605
 1      Kaufman County, Texas         64.466013
 2      Liberty County, Texas         60.775145
 3        Comal County, Texas         51.111689
 4         Hays County, Texas         46.649133
 ..                       ...               ...
 249    Sherman Count

### Migration to the county (ACS5)

In [53]:
# ACS 5-year migration change - only non-overlapping periods are valid
tx_counties_migration_acs5['pct_chg_migration_09_24'] = ((tx_counties_migration_acs5['total_in_migration_2024'] - tx_counties_migration_acs5['total_in_migration_2009']) / tx_counties_migration_acs5['total_in_migration_2009'] * 100)
tx_counties_migration_acs5['pct_chg_migration_14_24'] = ((tx_counties_migration_acs5['total_in_migration_2024'] - tx_counties_migration_acs5['total_in_migration_2014']) / tx_counties_migration_acs5['total_in_migration_2014'] * 100)
tx_counties_migration_acs5['pct_chg_migration_19_24'] = ((tx_counties_migration_acs5['total_in_migration_2024'] - tx_counties_migration_acs5['total_in_migration_2019']) / tx_counties_migration_acs5['total_in_migration_2019'] * 100)

# Replace inf values with NaN (for counties that had zero migration in the base year)
tx_counties_migration_acs5.replace([float('inf'), float('-inf')], float('nan'), inplace=True)

# Print ranked results
print_rankings(
    df=tx_counties_migration_acs5,
    variable_names=['pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

                            name  pct_chg_migration_09_24
0     Throckmorton County, Texas               411.111111
1           Oldham County, Texas               367.924528
2         Chambers County, Texas               187.765293
3         Hudspeth County, Texas               185.897436
4         Crockett County, Texas               178.160920
5            Mason County, Texas               148.333333
6          Runnels County, Texas               139.847716
7          Haskell County, Texas               124.844720
8          Coleman County, Texas               105.415162
9           Waller County, Texas               102.640374
10        Hansford County, Texas               102.040816
11          Martin County, Texas                98.175182
12          Kinney County, Texas                95.019157
13         Bastrop County, Texas                88.162064
14            Polk County, Texas                86.237685
15          Brooks County, Texas                83.216783
16            

{'pct_chg_migration_09_24':                            name  pct_chg_migration_09_24
 0    Throckmorton County, Texas               411.111111
 1          Oldham County, Texas               367.924528
 2        Chambers County, Texas               187.765293
 3        Hudspeth County, Texas               185.897436
 4        Crockett County, Texas               178.160920
 ..                          ...                      ...
 249       Roberts County, Texas               -87.037037
 250       Sherman County, Texas               -87.601078
 251      McMullen County, Texas               -97.191011
 252        Loving County, Texas              -100.000000
 253        Kenedy County, Texas                      NaN
 
 [254 rows x 2 columns],
 'pct_chg_migration_14_24':                          name  pct_chg_migration_14_24
 0          King County, Texas              3200.000000
 1        Kenedy County, Texas               800.000000
 2       Runnels County, Texas               327.601810

## ACS5 margins of error

In [54]:
smith_estimates_acs5 = tx_counties_acs5[tx_counties_acs5['name'] == 'Smith County, Texas'].copy()
smith_estimates_acs5

,name,population,total_families,family_households_with_child,median_age,population_under_18,lived_in_same_house,moved_within_county,moved_from_diff_county_same_state,moved_from_diff_state,...,population_under_18_moe,lived_in_same_house_moe,moved_within_county_moe,moved_from_diff_county_same_state_moe,moved_from_diff_state_moe,moved_from_abroad_moe,state,county_code,year,total_in_migration
185,"Smith County, Texas",197631,48076,23912,35.7,51096,155721,23260,10612,3891,...,0,2186,1915,1195,848,322,48,423,2009,15296
299,"Smith County, Texas",214735,53927,26596,36.0,54296,172408,23577,10885,4206,...,-555555555,1791,1568,1188,762,304,48,423,2014,15839
629,"Smith County, Texas",227449,53919,25385,36.6,56012,195995,14368,10144,3540,...,-555555555,2119,1645,1281,809,352,48,423,2019,14649
973,"Smith County, Texas",241740,57522,27097,37.5,58485,210107,14964,9982,3187,...,-555555555,2237,1753,1160,737,222,48,423,2024,13739


In [55]:
# Create migration confidence intervals dataframe for Smith County
migration_data_smith_acs5 = []

for year in smith_estimates_acs5['year'].unique():
    row = smith_estimates_acs5[smith_estimates_acs5['year'] == year].iloc[0]
    
    # Calculate total MOE
    total_moe = np.sqrt(
        row["moved_from_diff_county_same_state_moe"]**2 +
        row["moved_from_diff_state_moe"]**2 +
        row["moved_from_abroad_moe"]**2
    )
    
    migration_data_smith_acs5.append({
        'year': year,
        'total_in_migration': row['total_in_migration'],
        'moe': total_moe,
        'lower_bound': row['total_in_migration'] - total_moe,
        'upper_bound': row['total_in_migration'] + total_moe
    })

smith_migration_moes_acs5 = pd.DataFrame(migration_data_smith_acs5)
smith_migration_moes_acs5

,year,total_in_migration,moe,lower_bound,upper_bound
0,2009,15296,1500.270976,13795.729024,16796.270976
1,2014,15839,1443.746515,14395.253485,17282.746515
2,2019,14649,1555.424701,13093.575299,16204.424701
3,2024,13739,1392.139720,12346.860280,15131.139720


# Filter only to similarly-sized jurisdictions
We're going with MSAs and counties that had between 200K and 300K people in 2024. The Tyler metro area/Smith County had about 250K-ish people.

## ACS1

### Similarly sized MSAs

In [56]:
# filter msas
msa_pop_filtered = msa_tx_pop[(msa_tx_pop[2024] >= 200000) & (msa_tx_pop[2024] <= 300000)]
filtered_msa_codes = msa_pop_filtered['msa_code'].tolist()

# filter all msas dfs
msa_tx_pop_filtered = msa_tx_pop[msa_tx_pop['msa_code'].isin(filtered_msa_codes)].copy()
msa_tx_families_filtered = msa_tx_families[msa_tx_families['msa_code'].isin(filtered_msa_codes)].copy()
msa_tx_under18_filtered = msa_tx_under18[msa_tx_under18['msa_code'].isin(filtered_msa_codes)].copy()
msa_tx_family_households_with_child_filtered = msa_tx_family_households_with_child[msa_tx_family_households_with_child['msa_code'].isin(filtered_msa_codes)].copy()
msa_tx_migration_filtered = msa_tx_migration[msa_tx_migration['msa_code_'].isin(filtered_msa_codes)].copy()

In [57]:
print("Filtered MSAs - population change")
print_rankings(
    df=msa_tx_pop_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs - change in number of families")
print_rankings(
    df=msa_tx_families_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs - change in total population under 18")
print_rankings(
    df=msa_tx_under18_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs - change in family households with children under 18")
print_rankings(
    df=msa_tx_family_households_with_child_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs - migration change")
print_rankings(
    df=msa_tx_migration_filtered,
    name_col='name_',
    variable_names=['pct_chg_migration_05_24', 'pct_chg_migration_09_24',
                 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)   


Filtered MSAs - population change
year                                  name  pct_change_05_24
0     College Station-Bryan, TX Metro Area         58.756102
1                  Longview, TX Metro Area         48.176936
2                     Tyler, TX Metro Area         34.306203
3                    Laredo, TX Metro Area         23.182889
4                  Amarillo, TX Metro Area         22.182275
year                                  name  pct_change_09_24
0                  Longview, TX Metro Area         42.835736
1     College Station-Bryan, TX Metro Area         35.191783
2                     Tyler, TX Metro Area         21.706691
3                  Amarillo, TX Metro Area         13.274891
4                    Laredo, TX Metro Area         12.999196
year                                  name  pct_change_14_24
0                  Longview, TX Metro Area         35.869340
1     College Station-Bryan, TX Metro Area         20.344818
2                     Tyler, TX Metro Area         

{'pct_chg_migration_05_24':                                   name_  pct_chg_migration_05_24
 0  College Station-Bryan, TX Metro Area                79.185659
 1               Amarillo, TX Metro Area                53.519363
 2               Longview, TX Metro Area                20.921128
 3                  Tyler, TX Metro Area                 5.376472
 4                 Laredo, TX Metro Area               -49.005236,
 'pct_chg_migration_09_24':                                   name_  pct_chg_migration_09_24
 0  College Station-Bryan, TX Metro Area                20.908211
 1                  Tyler, TX Metro Area                11.042868
 2               Amarillo, TX Metro Area                 5.767439
 3               Longview, TX Metro Area                -8.985212
 4                 Laredo, TX Metro Area               -51.628923,
 'pct_chg_migration_14_24':                                   name_  pct_chg_migration_14_24
 0                  Tyler, TX Metro Area                44.

### Similarly sized counties

In [58]:
# Filter counties with population between 200K-300K in 2024
county_pop_filtered = county_tx_pop[(county_tx_pop[2024] >= 200000) & (county_tx_pop[2024] <= 300000)]
filtered_county_codes = county_pop_filtered['county_code'].tolist()

# Filter all county dataframes
county_tx_pop_filtered = county_tx_pop[county_tx_pop['county_code'].isin(filtered_county_codes)].copy()
county_tx_families_filtered = county_tx_families[county_tx_families['county_code'].isin(filtered_county_codes)].copy()
county_tx_under18_filtered = county_tx_under18[county_tx_under18['county_code'].isin(filtered_county_codes)].copy()
county_tx_family_households_with_child_filtered = county_tx_family_households_with_child[county_tx_family_households_with_child['county_code'].isin(filtered_county_codes)].copy()
tx_counties_migration_filtered = tx_counties_migration[tx_counties_migration['county_code'].isin(filtered_county_codes)].copy()

In [59]:
print("Filtered counties - population change")
print_rankings(
    df=county_tx_pop_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties - change in number of families")
print_rankings(
    df=county_tx_families_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties - change in total population under 18")
print_rankings(
    df=county_tx_under18_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties - change in family households with children under 18")
print_rankings(
    df=county_tx_family_households_with_child_filtered,
    variable_names=['pct_change_05_24', 'pct_change_09_24', 
                 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties - migration change")
print_rankings(
    df=tx_counties_migration_filtered,
    name_col='name',
    variable_names=['pct_chg_migration_05_24', 'pct_chg_migration_09_24',
                 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)   


Filtered counties - population change
year                     name  pct_change_05_24
0          Hays County, Texas        153.872033
1         Comal County, Texas        112.701226
2         Ellis County, Texas         76.747034
3        Brazos County, Texas         73.951583
4       Johnson County, Texas         46.460346
5         Smith County, Texas         34.306203
6      McLennan County, Texas         25.992861
7          Webb County, Texas         23.182889
8     Jefferson County, Texas          9.786391
year                     name  pct_change_09_24
0          Hays County, Texas         87.745668
1         Comal County, Texas         76.055883
2         Ellis County, Texas         53.151176
3        Brazos County, Texas         38.686164
4       Johnson County, Texas         34.108932
5         Smith County, Texas         21.706691
6      McLennan County, Texas         15.845538
7          Webb County, Texas         12.999196
8     Jefferson County, Texas          4.403524
ye

{'pct_chg_migration_05_24':                       name  pct_chg_migration_05_24
 0       Hays County, Texas               124.803569
 1     Brazos County, Texas                93.648208
 2      Comal County, Texas                51.468564
 3    Johnson County, Texas                50.633302
 4      Ellis County, Texas                48.392984
 5   McLennan County, Texas                21.644052
 6      Smith County, Texas                 5.376472
 7  Jefferson County, Texas               -12.765957
 8       Webb County, Texas               -49.005236,
 'pct_chg_migration_09_24':                       name  pct_chg_migration_09_24
 0       Hays County, Texas                61.698357
 1      Comal County, Texas                42.401035
 2     Brazos County, Texas                27.029915
 3    Johnson County, Texas                15.737005
 4      Smith County, Texas                11.042868
 5      Ellis County, Texas                10.685926
 6   McLennan County, Texas                -

## ACS5

### Similarly sized MSAs

In [60]:
# filter msas
msa_pop_filtered_acs5 = msa_tx_pop_acs5[(msa_tx_pop_acs5[2024] >= 200000) & (msa_tx_pop_acs5[2024] <= 300000)]
filtered_msa_codes_acs5 = msa_pop_filtered_acs5['msa_code'].tolist()

# filter all msas dfs
msa_tx_pop_filtered_acs5 = msa_tx_pop_acs5[msa_tx_pop_acs5['msa_code'].isin(filtered_msa_codes_acs5)].copy()
msa_tx_families_filtered_acs5 = msa_tx_families_acs5[msa_tx_families_acs5['msa_code'].isin(filtered_msa_codes_acs5)].copy()
msa_tx_under18_filtered_acs5 = msa_tx_under18_acs5[msa_tx_under18_acs5['msa_code'].isin(filtered_msa_codes_acs5)].copy()
msa_tx_family_households_with_child_filtered_acs5 = msa_tx_family_households_with_child_acs5[msa_tx_family_households_with_child_acs5['msa_code'].isin(filtered_msa_codes_acs5)].copy()
msa_tx_migration_filtered_acs5 = msa_tx_migration_acs5[msa_tx_migration_acs5['msa_code_'].isin(filtered_msa_codes_acs5)].copy()

In [61]:
print("Filtered MSAs (ACS5) - population change")
print_rankings(
    df=msa_tx_pop_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs (ACS5) - change in number of families")
print_rankings(
    df=msa_tx_families_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs (ACS5) - change in total population under 18")
print_rankings(
    df=msa_tx_under18_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs (ACS5) - change in family households with children under 18")
print_rankings(
    df=msa_tx_family_households_with_child_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered MSAs (ACS5) - migration change")
print_rankings(
    df=msa_tx_migration_filtered_acs5,
    name_col='name_',
    variable_names=['pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

Filtered MSAs (ACS5) - population change
year                                  name  pct_change_09_24
0                  Longview, TX Metro Area         43.385555
1     College Station-Bryan, TX Metro Area         36.541801
2                     Tyler, TX Metro Area         22.318867
3                    Laredo, TX Metro Area         16.559829
4                  Amarillo, TX Metro Area         12.427902
year                                  name  pct_change_14_24
0                  Longview, TX Metro Area         34.278833
1     College Station-Bryan, TX Metro Area         18.281375
2                     Tyler, TX Metro Area         12.575966
3                  Amarillo, TX Metro Area          5.788638
4                    Laredo, TX Metro Area          3.785780
year                                  name  pct_change_19_24
0     College Station-Bryan, TX Metro Area          7.869658
1                     Tyler, TX Metro Area          6.283167
2                  Amarillo, TX Metro Area  

{'pct_chg_migration_09_24':                                   name_  pct_chg_migration_09_24
 0  College Station-Bryan, TX Metro Area                20.642996
 1               Longview, TX Metro Area                 4.792725
 2                  Tyler, TX Metro Area               -10.179132
 3               Amarillo, TX Metro Area               -19.750394
 4                 Laredo, TX Metro Area               -45.106875,
 'pct_chg_migration_14_24':                                   name_  pct_chg_migration_14_24
 0  College Station-Bryan, TX Metro Area                10.096640
 1               Longview, TX Metro Area                -1.902759
 2                  Tyler, TX Metro Area               -13.258413
 3               Amarillo, TX Metro Area               -14.276469
 4                 Laredo, TX Metro Area               -45.484796,
 'pct_chg_migration_19_24':                                   name_  pct_chg_migration_19_24
 0  College Station-Bryan, TX Metro Area                 5.

### Similarly-sized coutnies (ACS5)

In [62]:
# Filter counties with population between 200K-300K in 2024 (ACS5)
county_pop_filtered_acs5 = county_tx_pop_acs5[(county_tx_pop_acs5[2024] >= 200000) & (county_tx_pop_acs5[2024] <= 300000)]
filtered_county_codes_acs5 = county_pop_filtered_acs5['county_code'].tolist()

# Filter all county dfs
county_tx_pop_filtered_acs5 = county_tx_pop_acs5[county_tx_pop_acs5['county_code'].isin(filtered_county_codes_acs5)].copy()
county_tx_families_filtered_acs5 = county_tx_families_acs5[county_tx_families_acs5['county_code'].isin(filtered_county_codes_acs5)].copy()
county_tx_under18_filtered_acs5 = county_tx_under18_acs5[county_tx_under18_acs5['county_code'].isin(filtered_county_codes_acs5)].copy()
county_tx_family_households_with_child_filtered_acs5 = county_tx_family_households_with_child_acs5[county_tx_family_households_with_child_acs5['county_code'].isin(filtered_county_codes_acs5)].copy()
tx_counties_migration_filtered_acs5 = tx_counties_migration_acs5[tx_counties_migration_acs5['county_code'].isin(filtered_county_codes_acs5)].copy()

In [63]:
print("Filtered counties (ACS5) - population change")
print_rankings(
    df=county_tx_pop_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties (ACS5) - change in number of families")
print_rankings(
    df=county_tx_families_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties (ACS5) - change in total population under 18")
print_rankings(
    df=county_tx_under18_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties (ACS5) - change in family households with children under 18")
print_rankings(
    df=county_tx_family_households_with_child_filtered_acs5,
    variable_names=['pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']
)

print("Filtered counties (ACS5) - migration change")
print_rankings(
    df=tx_counties_migration_filtered_acs5,
    name_col='name',
    variable_names=['pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']
)

Filtered counties (ACS5) - population change
year                     name  pct_change_09_24
0          Hays County, Texas         90.023414
1         Ellis County, Texas         49.745694
2        Brazos County, Texas         41.386493
3         Smith County, Texas         22.318867
4          Webb County, Texas         16.559829
5      McLennan County, Texas         16.507494
6     Jefferson County, Texas          4.616031
year                     name  pct_change_14_24
0          Hays County, Texas         57.642157
1         Ellis County, Texas         38.014982
2        Brazos County, Texas         20.233311
3         Smith County, Texas         12.575966
4      McLennan County, Texas         10.992128
5          Webb County, Texas          3.785780
6     Jefferson County, Texas          0.559283
year                     name  pct_change_19_24
0          Hays County, Texas         25.904783
1         Ellis County, Texas         22.666483
2        Brazos County, Texas          8.66

{'pct_chg_migration_09_24':                       name  pct_chg_migration_09_24
 0       Hays County, Texas                41.339738
 1     Brazos County, Texas                21.721357
 2      Ellis County, Texas                20.153518
 3   McLennan County, Texas                 4.169439
 4  Jefferson County, Texas                -2.519005
 5      Smith County, Texas               -10.179132
 6       Webb County, Texas               -45.106875,
 'pct_chg_migration_14_24':                       name  pct_chg_migration_14_24
 0      Ellis County, Texas                52.765127
 1       Hays County, Texas                41.094978
 2   McLennan County, Texas                15.101391
 3     Brazos County, Texas                10.936494
 4  Jefferson County, Texas                -8.407990
 5      Smith County, Texas               -13.258413
 6       Webb County, Texas               -45.484796,
 'pct_chg_migration_19_24':                       name  pct_chg_migration_19_24
 0      Ellis Co

# Export for graphics

In [64]:
# Create output csvs for graphics 
# Population change - ACS5

# population change -- want county code, county name, percentage change, and true/false for whether the county is between 200k to 300k in population
county_pop_export = county_tx_pop_acs5[['county_code', 'name', 'pct_change_09_24', 'pct_change_14_24', 'pct_change_19_24']].copy()
county_pop_export['name'] = county_pop_export['name'].str.replace(' County, Texas', '', regex=False)

# create a column for whether the county is between 200k to 300k in population in 2024
county_pop_export['pop_200k_300k'] = county_tx_pop_acs5[2024].between(200000, 300000)

# create output directory if it doesn't exist
import os
os.makedirs('output', exist_ok=True)
county_pop_export.to_csv('output/county_population_change.csv', index=False)

print("Exported to output/county_population_change.csv")

Exported to output/county_population_change.csv


In [65]:
# Create output csvs for graphics
# Change in migration - ACS5

# migration -- want county code, county name, percentage change, and true/false for whether the county is between 200k to 300k in population
county_migration_export = tx_counties_migration_acs5[['county_code', 'name', 'pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']].copy()
county_migration_export['name'] = county_migration_export['name'].str.replace(' County, Texas', '', regex=False)

# create a column for whether the county is between 200k to 300k in population in 2024
county_migration_export['pop_200k_300k'] = tx_counties_migration_acs5[2024].between(200000, 300000)

county_migration_export.to_csv('output/county_migration_change.csv', index=False)

print("Exported to output/county_migration_change.csv")

KeyError: 2024

In [ ]:
# Create output csvs for graphics
# Change in # of families - ACS5

# migration -- want county code, county name, percentage change, and true/false for whether the county is between 200k to 300k in population
county_family_export = county_tx_families_acs5[['county_code', 'name', 'pct_chg_migration_09_24', 'pct_chg_migration_14_24', 'pct_chg_migration_19_24']].copy()
county_family_export['name'] = county_family_export['name'].str.replace(' County, Texas', '', regex=False)

# create a column for whether the county is between 200k to 300k in population in 2024
county_family_export['pop_200k_300k'] = county_tx_families_acs5[2024].between(200000, 300000)
county_family_export.to_csv('output/county_families_change.csv', index=False)

print("Exported to output/county_families_change.csv")

In [ ]:
# i want to output a spreadsheet of moes for migration estimates for acs5 and acs1
